# Reproducing Watson et al. (2016) from dandiset 000041

**Paper.** Watson BO, Levenstein D, Greene JP, Gelinas JN, Buzsáki G (2016). Network homeostasis
and state dynamics of neocortical sleep. *Neuron* 90:839-852.
[doi:10.1016/j.neuron.2016.03.036](https://doi.org/10.1016/j.neuron.2016.03.036)

**Text used.** The full text of the NIH author manuscript in PubMed Central (PMC4873379): Results,
Experimental Procedures and figure legends. The Supplemental Experimental Procedures and the
supplementary figures could not be retrieved, and Table 1 is not in the manuscript text, so methods
described only there (UP/DOWN state and spindle detection, the autocorrelogram decay fit) are not
available to this attempt.

**Data.** [DANDI:000041 version 0.250624.0419](https://dandiarchive.org/dandiset/000041/0.250624.0419),
streamed from the archive. The paper cites no dandiset version (it points to CRCNS). There are two
published versions, 0.210812.1515 and 0.250624.0419; they hold the same 22 assets (same asset ids,
sizes and modification dates) and differ only in dandiset metadata (contributor affiliations and
identifiers, a data curator, the related-resource record). The most recent published version is
pinned, as no earlier one matches the paper better.

**Code.** The paper links no analysis code, and the dandiset links none. The NWB conversion code,
[catalystneuro/buzsaki-lab-to-nwb](https://github.com/catalystneuro/buzsaki-lab-to-nwb) at commit
5f6a3750fffd999c4757d8a198cba6007f85fb12 (`buzsaki_lab_to_nwb/watson/`), was read: the state table
comes from each session's `SleepState.states.mat` with only WAKE, NREM and REM intervals, and the
units of Dino_072114 were read from a `spikes.cellinfo.mat` file with a hard-coded 20 kHz rate.

| Claim | Tested or blocked |
|---|---|
| dataset-composition | tested |
| fig2a-median-rates | tested |
| fig2a-skewed-wide-distribution | tested |
| fig2a-fraction-above-2hz | tested |
| fig2b-wake-nrem-slope | tested |
| fig2-burst-index-by-state | tested |
| fig2c-acg-peaks | tested |
| fig2c-acg-decay | blocked: methods underspecified |
| fig2f-rate-group-correlations | tested |
| ma-statistics | tested |
| ma-fraction-over-nrem-episodes | tested |
| packet-duration | tested |
| wake-sleep-cycles | tested |
| fig3b-population-decline | tested |
| fig3-first-last-packet | tested |
| fig3d-first-last-slope | tested |
| fig3e-wake-before-after | tested |
| fig3b-sextiles-over-sleep | tested |
| fig3b-cv-over-sleep | tested |
| fig4iv-sextiles-within-nrem | tested |
| fig4v-cv-within-nrem | tested |
| fig4vi-burst-within-nrem | tested |
| fig4i-band-power-within-nrem | tested |
| fig4-delta-packet-length | tested |
| fig4cd-packet-rate-lfp | tested |
| fig5b-nrem-between-ma | tested |
| fig5b-wake-between-packets | tested |
| fig5b-ma-between-packets | tested |
| fig5b-rem-between-packets | tested |
| fig5c-rem-duration | tested |
| fig5c-sigma-ma | tested |
| fig5c-delta-change | tested |
| figs5-within-wake-rem | tested |
| fig5c-wake-movement | blocked: data missing |
| updown-durations | blocked: methods underspecified |
| fig3c-slow-oscillation-spindles-over-sleep | blocked: methods underspecified |
| fig4ii-iii-updown-spindles-within-packets | blocked: methods underspecified |
| fig2de-up-state-sequences | blocked: methods underspecified |
| fig3f-4e-within-up-rates | blocked: methods underspecified |

The criteria for each claim are in `claims/000041/10.1016-j.neuron.2016.03.036.yaml`, written before
the analysis. The definitions used throughout (microarousals, packets, episodes, SLEEP, WAKE-SLEEP
cycles, sextiles, time courses) are in that file's opening comment and in `sleepstates.py`.

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt

import sleepstates as ss

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)

# Streams every usable file: units, states, and the LFP power of the reference channel in
# Non-REM epochs. About two minutes with six processes.
sessions, excluded = ss.load_all(workers=6)
edges, sext = ss.wake_sextiles(sessions)
print(len(sessions), "sessions loaded; excluded:", excluded)

## Data check

In [ ]:
import collections
import h5py, remfile

urls = dict(ss.asset_urls())
rows = []
for s in sessions:
    dur = s.stops - s.starts
    spk_all = np.concatenate([x for x in s.spikes if len(x)])
    rows.append(dict(
        subject=s.subject, session=s.session_id,
        pE=int((s.cell_type == "excitatory").sum()), pI=int((s.cell_type == "inhibitory").sum()),
        unknown=int((s.cell_type == "unknown").sum()),
        states_min=round((s.stops[-1] - s.starts[0]) / 60, 1),
        spikes_first=round(spk_all.min(), 2), spikes_last_min=round(spk_all.max() / 60, 1),
        lfp_min=round(s.lfp_samples / ss.LFP_RATE / 60, 1),
        WAKE_min=round(dur[s.state == "WAKE"].sum() / 60, 1), MA_n=int((s.state == "MA").sum()),
        NREM_min=round(dur[s.state == "NREM"].sum() / 60, 1), REM_min=round(dur[s.state == "REM"].sum() / 60, 1),
        epochs=len(s.labels), lfp_packets=len(s.lfp)))
tab = pd.DataFrame(rows)
print(tab.to_string())
print("\nTotals over the 21 usable sessions:", tab[["pE", "pI", "unknown", "MA_n"]].sum().to_dict())
print("Subjects:", tab.subject.nunique(), "| sessions per subject:", tab.groupby("subject").size().to_dict())
print("pE cells per subject:", tab.groupby("subject").pE.sum().to_dict())
print("State labels in the files:", sorted(set(np.concatenate([s.labels for s in sessions]))))

# Units, ranges and timing of what is used.
for s in sessions:
    dur = s.stops - s.starts
    assert np.all(np.diff(s.starts) > 0) and np.all(dur >= 0)
    assert all(np.all(np.diff(x) >= 0) for x in s.spikes)
gaps = np.concatenate([s.starts[1:] - s.stops[:-1] for s in sessions])
allstart = np.concatenate([s.starts for s in sessions])
print("\nState table: start times are whole seconds:", bool(np.all(allstart == np.round(allstart))),
      "| gaps between consecutive epochs (s), values and counts:",
      dict(zip(*[x.tolist() for x in np.unique(gaps, return_counts=True)])))
print("State durations (s): min", min((s.stops - s.starts).min() for s in sessions),
      "max", max((s.stops - s.starts).max() for s in sessions))
print("Spikes after the end of the state table (fraction of spikes), per session:",
      {s.session_id: round(float(np.mean(np.concatenate(s.spikes) > s.stops[-1])), 3) for s in sessions})
isi_viol = []
for s in sessions:
    for u in s.pe:
        x = s.spikes[u]
        if len(x) > 1:
            isi_viol.append(np.mean(np.diff(x) < 0.001))
isi_viol = np.array(isi_viol)
print("pE units: fraction of intervals under 1 ms, median %.4f, 95th percentile %.4f, max %.4f"
      % (np.median(isi_viol), np.percentile(isi_viol, 95), isi_viol.max()))
print("pE spike counts per unit: min", min(len(s.spikes[u]) for s in sessions for u in s.pe),
      "median", int(np.median([len(s.spikes[u]) for s in sessions for u in s.pe])))
print("LFP reference column per session:", {s.session_id: s.lfp_channel for s in sessions})

# The excluded file: spike times are whole seconds divided by 20000.
p = list(excluded)[0]
f = h5py.File(remfile.File(urls[p]), "r")
t = f["units/spike_times"][()]
print("\n%s: %d spikes, %d distinct values, max %.4f s; max x 20000 = %.0f; times x 20000 all integers: %s;"
      " LFP lasts %.0f s, states end at %.0f s" % (
          p, len(t), len(np.unique(t)), t.max(), t.max() * 20000, bool(np.allclose(t * 20000, np.round(t * 20000), atol=1e-6)),
          f["processing/ecephys/LFP/lfp/data"].shape[0] / 1250, f["processing/behavior/states/stop_time"][()].max()))
print("LFP data attributes: unit %r, conversion %r" % (
    f["processing/ecephys/LFP/lfp/data"].attrs["unit"], f["processing/ecephys/LFP/lfp/data"].attrs["conversion"]))
el = f["general/extracellular_ephys/electrodes"]
print("electrode location values:", set(ss._str(el["location"][()])),
      "| electrode group locations:", sorted({f["general/extracellular_ephys"][g].attrs.get("location", "")
                                             for g in f["general/extracellular_ephys"] if g.startswith("shank")}))
print("subject age, weight:", f["general/subject/age"][()], f["general/subject/weight"][()])

# Recordings deposited more than once: identical spike trains or state tables across files.
sig_units = collections.defaultdict(list)
for s in sessions:
    for u, x in enumerate(s.spikes):
        if len(x) >= 5:
            sig_units[(len(x), tuple(np.round(x[:5], 5)))].append((s.session_id, u))
dups = [v for v in sig_units.values() if len({a for a, _ in v}) > 1]
print("\nUnits with identical spike trains in different files:", dups if dups else "none")
st_sig = collections.defaultdict(list)
for s in sessions:
    st_sig[(len(s.starts), tuple(s.starts[:10]), tuple(s.stops[-3:]))].append(s.session_id)
print("Identical state tables in different files:", [v for v in st_sig.values() if len(v) > 1] or "none")
same_counts = collections.Counter(tuple(sorted(len(x) for x in s.spikes)) for s in sessions)
print("Files with identical sets of per-unit spike counts:", [k for k, v in same_counts.items() if v > 1] or "none")

In [ ]:
# LFP: the lag-1 sample autocorrelation of the reference channel, from the raw samples kept at
# load time. LFP at 1250 Hz is smooth (near 1); scrambled samples are not.
lag1 = {s.session_id: float(np.mean([np.corrcoef(x[:-1].astype(float), x[1:].astype(float))[0, 1] for _, x in s.lfp_raw]))
        for s in sessions}
print("LFP lag-1 autocorrelation per session:", {k: round(v, 3) for k, v in lag1.items()})
lfp_bad = [s.session_id for s in sessions if lag1[s.session_id] < 0.8]
print("Sessions whose LFP is left out of the LFP analyses (lag-1 autocorrelation under 0.8):", lfp_bad)
for s in sessions:
    if s.session_id in lfp_bad:
        print("  %s: LFP has %d columns for %d electrodes and lasts %.1f min; states end at %.1f min" % (
            s.session_id, len(h5py.File(remfile.File(urls[s.path]), "r")["processing/ecephys/LFP/lfp/electrodes"][()]),
            len(h5py.File(remfile.File(urls[s.path]), "r")["general/extracellular_ephys/electrodes/id"][()]),
            s.lfp_samples / ss.LFP_RATE / 60, s.stops[-1] / 60))
        s.lfp = {}

### Do the spikes, the states and the LFP share one clock?

The state labels are tested against the spikes by a contrast that an offset would destroy: the
fraction of 50 ms bins in which no pE cell fires (DOWN states make this high in Non-REM) in Non-REM
minus WAKE, with the state table shifted in time. If the clocks agree, the contrast is largest at
zero shift. The LFP is tested against the spikes inside Non-REM packets: the correlation between
each 2 s window's delta power and the pE spike count in the same window, with the spikes shifted.

In [ ]:
shifts = np.array([-300, -120, -60, -20, -5, 0, 5, 20, 60, 120, 300])
clock = []
for s in sessions:
    pop = np.sort(np.concatenate([s.spikes[u] for u in s.pe]))
    edges50 = np.arange(0, s.stops[-1], 0.05)
    silent = np.histogram(pop, edges50)[0] == 0
    centres = edges50[:-1] + 0.025
    row = {"session": s.session_id, "pE": len(s.pe)}
    for d in shifts:
        k = np.searchsorted(s.starts + d, centres, side="right") - 1
        ok = (k >= 0) & (centres < (s.stops + d)[np.clip(k, 0, None)])
        lab = np.where(ok, s.state[np.clip(k, 0, None)], "none")
        row[int(d)] = silent[lab == "NREM"].mean() - silent[lab == "WAKE"].mean()
    # LFP against spikes: cross-correlation of LFP and pE spike count in 8 ms bins (10 samples)
    lags = np.arange(-125, 126)
    acc = np.zeros(len(lags)); nseg = 0
    for t0, x in s.lfp_raw:
        nb = len(x) // 10
        lf = x[:nb * 10].reshape(nb, 10).mean(axis=1)
        sp = np.diff(np.searchsorted(pop, t0 + np.arange(nb + 1) * 0.008))
        lf = (lf - lf.mean()) / lf.std(); sp = (sp - sp.mean()) / (sp.std() or 1)
        acc += np.array([np.mean(lf[max(0, -L):nb - max(0, L)] * sp[max(0, L):nb - max(0, -L)]) for L in lags]); nseg += 1
    cc = acc / nseg
    j = np.argmax(np.abs(cc))
    row.update({"lfp seconds": round(sum(len(x) for _, x in s.lfp_raw) / ss.LFP_RATE), "xcorr extreme lag ms": lags[j] * 8,
                "xcorr extreme": round(cc[j], 3), "xcorr at -1 s": round(cc[0], 3), "xcorr at +1 s": round(cc[-1], 3)})
    clock.append(row)
clock = pd.DataFrame(clock)
best = clock[[int(d) for d in shifts]].idxmax(axis=1)
clock["best shift"] = best
clock["labels consistent"] = (best.abs() <= 60) & (clock[0] > clock[[-300, 300]].max(axis=1))
clock["lfp consistent"] = (clock["xcorr extreme lag ms"].abs() <= 50) & (clock["xcorr extreme"].abs() > 3 * clock[["xcorr at -1 s", "xcorr at +1 s"]].abs().max(axis=1))
print("Contrast of silent-bin fraction, NREM minus WAKE, by shift of the state table (s), and the LFP-spike cross-correlation:")
print(clock.round(3).to_string())
print("\nState labels consistent with the spikes (largest contrast within 60 s of zero shift and above the contrast at +-300 s):",
      int(clock["labels consistent"].sum()), "of", len(clock), "| not established:", list(clock.session[~clock["labels consistent"]]))
print("LFP consistent with the spikes (largest |r| within 50 ms of zero lag and over 3 times |r| at +-1 s):",
      int(clock["lfp consistent"].sum()), "of", len(clock), "| not established:", list(clock.session[~clock["lfp consistent"]]))
print("Shift with the largest contrast, counts:", best.value_counts().sort_index().to_dict())

The data check against the description:

- The dandiset description and the paper give 11 rats and 27 sessions; the files hold 10 rats
  (the archive's own summary says 10 subjects) and 22 sessions, one of which (Dino-072114) has
  unusable spike times. The description's unit counts (1360 recorded, 1121 stable, 995 pE, 126 pI)
  are those of the paper; the files hold 945 units: 826 excitatory, 96 inhibitory and 23 unknown.
- The state table has only Awake, Non-REM and REM, at 1 s resolution with 1 s gaps between epochs.
  Microarousals, which the paper scores as a state of their own, are not labelled; they are derived
  here from the Awake epochs under 40 s between two Non-REM epochs.
- LFP data are stored as int16 with unit "volts" and conversion NaN, so they cannot be put in volts;
  analyses use log power in raw units, normalized within session. The electrode `location` column
  is "unknown" for every electrode, while the electrode groups carry a region.
- Subject age and weight are the same range in every file ("3-7 months", "250-500g"); the dandiset
  description says 4-7 months and the paper 3-7 months.
- The clock checks above are printed per session; the outcome of each is stated in the summary.

## dataset-composition

> The recordings come from 11 rats and 27 sessions of 294 plus or minus 137 min, yielding 995
> putative pyramidal cells and 126 putative interneurons.

In [ ]:
n_files = len(sessions) + len(excluded)
dep = dict(rats=tab.subject.nunique(), sessions=n_files)
# the excluded file's units are counted from its own table
ct_ex = collections.Counter(ss._str(f["units/cell_type"][()]))
st_ex = f["processing/behavior/states"]
dur_ex = (st_ex["stop_time"][()].max() - st_ex["start_time"][()].min()) / 60
durs = list(tab.states_min) + [dur_ex]
dep.update(mean_duration_min=np.mean(durs), sd_duration_min=np.std(durs, ddof=1),
           pE=int(tab.pE.sum() + ct_ex["excitatory"]), pI=int(tab.pI.sum() + ct_ex["inhibitory"]),
           unknown=int(tab.unknown.sum() + ct_ex["unknown"]))
print("As deposited (22 files):", {k: (round(v, 1) if isinstance(v, float) else v) for k, v in dep.items()})
print("Usable (21 sessions): rats %d, sessions %d, pE %d, pI %d" % (tab.subject.nunique(), len(tab), tab.pE.sum(), tab.pI.sum()))
print("Mean spike span of the usable sessions: %.1f min" % tab.spikes_last_min.mean())
exp = dict(rats=(11, 1.1), sessions=(27, 2.7), mean_duration_min=(294, 137), pE=(995, 99.5), pI=(126, 12.6))
ok = {k: abs(dep[k] - v) <= t for k, (v, t) in exp.items()}
print("Within tolerance:", ok, "->", sum(ok.values()), "of 5")
print("pE cells per session: mean %.1f, range %d to %d (paper: mean 36.9, range 10 to 104)" % (
    tab.pE.mean(), tab.pE.min(), tab.pE.max()))
print("pI cells per session: mean %.2f, range %d to %d (paper: mean 4.67, range 0 to 13)" % (
    tab.pI.mean(), tab.pI.min(), tab.pI.max()))

**Result: not reproduced.** Two of the five values are within tolerance: 10 rats (11 expected) and a mean session duration of 286.8 min (294 plus or minus 137). The files hold 22 sessions (27 expected), 826 pE cells (995) and 96 pI cells (126); in the 21 usable sessions 812 pE and 89 pI. Per session the pE counts are close to the paper's (mean 38.7, range 10 to 105, against 36.9 and 10 to 104), so the shortfall comes from the missing sessions and rat.

## fig2a-median-rates

> Median firing rates of pE cells are 0.76 Hz in WAKE, 0.69 Hz in nonREM and 0.88 Hz in REM; of pI
> cells 5.59, 4.69 and 4.25 Hz.

In [ ]:
def cell_table(wake_states):
    out = []
    for s in sessions:
        for cls, units in (("pE", s.pe), ("pI", s.pi)):
            r = {lab: ss.state_rates(s, units, sts) for lab, sts in
                 (("WAKE", wake_states), ("NREM", ["NREM"]), ("REM", ["REM"]))}
            for k, u in enumerate(units):
                out.append(dict(subject=s.subject, session=s.session_id, unit=u, cls=cls,
                                WAKE=r["WAKE"][k], NREM=r["NREM"][k], REM=r["REM"][k]))
    return pd.DataFrame(out)

cells_a = cell_table(["WAKE"])
cells_b = cell_table(["WAKE", "MA"])
expected = {("pE", "WAKE"): 0.76, ("pE", "NREM"): 0.69, ("pE", "REM"): 0.88,
            ("pI", "WAKE"): 5.59, ("pI", "NREM"): 4.69, ("pI", "REM"): 4.25}
for name, cells in (("(a) WAKE without MA", cells_a), ("(b) WAKE with MA", cells_b)):
    med = cells.groupby("cls")[["WAKE", "NREM", "REM"]].median()
    sd = cells.groupby("cls")[["WAKE", "NREM", "REM"]].std()
    n_in = sum(abs(med.loc[c, s_] - v) <= 0.1 * v for (c, s_), v in expected.items())
    print(name, "| medians (Hz):\n", med.round(3).to_string(), "\nSD:\n", sd.round(2).to_string(),
          "\nwithin 10%:", n_in, "of 6\n")
print("n cells:", cells_a.cls.value_counts().to_dict())
print("Per subject, pE medians (reading a):")
print(cells_a[cells_a.cls == "pE"].groupby("subject")[["WAKE", "NREM", "REM"]].median().round(3).to_string())
print("\nPer subject, pI medians (reading a):")
print(cells_a[cells_a.cls == "pI"].groupby("subject")[["WAKE", "NREM", "REM"]].agg(["median", "size"]).round(2).to_string())

**Result: partially reproduced.** Three of six medians are within tolerance under both readings, all three pE values: 0.755, 0.662 and 0.825 Hz in WAKE, nonREM and REM under reading (a). The pI medians are 3.92, 2.73 and 3.06 Hz, below the paper's 5.59, 4.69 and 4.25; the pI sample is 89 cells, and its per-subject medians range from 0.07 to 13.4 Hz across subjects and states.

## fig2a-skewed-wide-distribution

> The distribution of pE firing rates is strongly positively skewed and spans three orders of
> magnitude, from 0.05 to 5 Hz.

In [ ]:
pe_a = cells_a[cells_a.cls == "pE"]
for state in ("WAKE", "NREM", "REM"):
    r = pe_a[state].values
    lo, hi = np.percentile(r, [2.5, 97.5])
    pos = r[r > 0]
    print("%s: skewness %.2f, skewness of log10 (rates > 0) %.2f, 2.5th pct %.4f Hz, 97.5th pct %.3f Hz, "
          "min %.4f, max %.2f, zero rates %d of %d" % (state, st.skew(r), st.skew(np.log10(pos)), lo, hi,
                                                        r.min(), r.max(), (r == 0).sum(), len(r)))
r = pe_a.WAKE.values
lo, hi = np.percentile(r, [2.5, 97.5])
print("\nWAKE: skewness > 1:", st.skew(r) > 1, "| 2.5th pct within 0.025-0.1 Hz:", 0.025 <= lo <= 0.1,
      "| 97.5th pct within 2.5-10 Hz:", 2.5 <= hi <= 10)
fig, ax = plt.subplots(figsize=(5, 3.5))
for state, c in (("WAKE", "k"), ("NREM", "tab:blue"), ("REM", "tab:red")):
    x = np.sort(pe_a[state].values)
    x = x[x > 0]
    ax.plot(x, np.arange(1, len(x) + 1) / len(x), color=c, label=state)
for e in edges[1:-1]:
    ax.axvline(e, color="0.7", lw=0.5)
ax.set_xscale("log"); ax.set_xlabel("rate (Hz)"); ax.set_ylabel("cumulative fraction of pE cells"); ax.legend()
ax.set_title("pE rates by state (zero rates not drawn)")
plt.show()
print("Rates drawn: from %.4f to %.2f Hz; zero rates not drawn: WAKE %d, NREM %d, REM %d" % (
    min(pe_a[s_][pe_a[s_] > 0].min() for s_ in ("WAKE", "NREM", "REM")),
    max(pe_a[s_].max() for s_ in ("WAKE", "NREM", "REM")),
    (pe_a.WAKE == 0).sum(), (pe_a.NREM == 0).sum(), (pe_a.REM == 0).sum()))
print("WAKE-rate sextile edges (Hz):", np.round(edges, 4))

**Result: partially reproduced.** WAKE rates are strongly skewed (skewness 4.57) and the 97.5th percentile, 7.17 Hz, is within a factor of 2 of 5 Hz, but the 2.5th percentile, 0.0124 Hz, is about four times below 0.05 Hz, so the distribution spans more than the paper's range at the low end. On a log scale the rates are skewed to the left (skewness of log10 rates -0.74), which agrees with the paper's description of a heavier tail toward low rates.

## fig2a-fraction-above-2hz

> The fraction of pE cells firing above 2 Hz is 19.2% in WAKE, 19.2% in REM and 8.8% in nonREM
> (chi-square, p < 1e-10).

In [ ]:
frac = {s_: 100 * np.mean(pe_a[s_] > 2) for s_ in ("WAKE", "REM", "NREM")}
print("Percent of pE cells above 2 Hz:", {k: round(v, 2) for k, v in frac.items()})
for other in ("WAKE", "REM"):
    a = (pe_a.NREM > 2).sum(); b = (pe_a[other] > 2).sum(); n = len(pe_a)
    chi2, p, _, _ = st.chi2_contingency([[a, n - a], [b, n - b]])
    print("NREM against %s: chi2 = %.1f, p = %.2g" % (other, chi2, p))
print("Within tolerance:", {k: abs(frac[k] - v) <= 0.1 * v for k, v in (("WAKE", 19.2), ("REM", 19.2), ("NREM", 8.8))})
print("\nPer subject, percent above 2 Hz:")
print((pe_a.groupby("subject")[["WAKE", "REM", "NREM"]].agg(lambda x: 100 * np.mean(x > 2))).round(1).to_string())

**Result: not reproduced.** The fractions above 2 Hz are 17.0% in WAKE (outside 17.3 to 21.1), 18.7% in REM and 9.1% in nonREM (both within tolerance). nonREM is lower than WAKE (chi-square p = 3.5e-06) and REM (p = 3.4e-08), but neither test reaches the paper's p < 1e-10. The direction of the effect is as in the paper.

## fig2b-wake-nrem-slope

> Per-cell nonREM and WAKE rates are highly correlated (R = 0.82), and the slope of their relation
> is below 1 (95% confidence interval 0.66 to 0.71).

In [ ]:
def ols(x, y):
    res = st.linregress(x, y)
    t = st.t.ppf(0.975, len(x) - 2)
    return res.slope, res.slope - t * res.stderr, res.slope + t * res.stderr, res.rvalue, res.pvalue, len(x)

keep = (pe_a.WAKE > 0) & (pe_a.NREM > 0)
fits = {"(a) log10": ols(np.log10(pe_a.WAKE[keep]), np.log10(pe_a.NREM[keep])),
        "(b) Hz": ols(pe_a.WAKE, pe_a.NREM),
        "(a) reverse, WAKE on NREM": ols(np.log10(pe_a.NREM[keep]), np.log10(pe_a.WAKE[keep])),
        "(b) reverse": ols(pe_a.NREM, pe_a.WAKE)}
for k, (b, lo, hi, r, p, n) in fits.items():
    print("%s: slope %.3f (95%% CI %.3f to %.3f), R = %.3f, p = %.2g, n = %d" % (k, b, lo, hi, r, p, n))
print("Dropped for zero rate in the log reading:", (~keep).sum())
for k in ("(a) log10", "(b) Hz"):
    b, lo, hi, r, p, n = fits[k]
    print(k, "| R within 0.738-0.902:", 0.738 <= r <= 0.902, "| slope within 0.594-0.781:", 0.594 <= b <= 0.781,
          "| CI excludes 1:", hi < 1)
print("\nPer subject, reading (a):")
for subj, g in pe_a[keep].groupby("subject"):
    b, lo, hi, r, p, n = ols(np.log10(g.WAKE), np.log10(g.NREM))
    print("  %s: slope %.2f (%.2f to %.2f), R %.2f, n %d" % (subj, b, lo, hi, r, n))
fig, ax = plt.subplots(figsize=(4, 4))
ax.loglog(pe_a.WAKE[keep], pe_a.NREM[keep], ".", ms=3)
ax.plot([1e-3, 30], [1e-3, 30], "k--", lw=0.8)
ax.set_xlabel("WAKE rate (Hz)"); ax.set_ylabel("nonREM rate (Hz)"); ax.set_title("pE cells")
plt.show()
print("Plotted: WAKE %.4f to %.2f Hz, NREM %.4f to %.2f Hz; %d cells with a zero rate not drawn" % (
    pe_a.WAKE[keep].min(), pe_a.WAKE[keep].max(), pe_a.NREM[keep].min(), pe_a.NREM[keep].max(), (~keep).sum()))
# Control for regression to the mean: each cell's WAKE epochs are split alternately into two halves.
# The slope of one half on the other shows what measurement noise alone does to a slope, and the
# reliability of the whole WAKE rate (Spearman-Brown) gives the slope corrected for attenuation.
half = []
for s in sessions:
    w_ = np.where(s.state == "WAKE")[0]
    half += list(zip(ss.rates(s.spikes, s.pe, s.starts[w_[0::2]], s.stops[w_[0::2]]),
                     ss.rates(s.spikes, s.pe, s.starts[w_[1::2]], s.stops[w_[1::2]])))
half = np.array(half)
kh = (half > 0).all(axis=1)
b, lo, hi, r_h, p, n = ols(np.log10(half[kh, 0]), np.log10(half[kh, 1]))
rel = 2 * r_h / (1 + r_h)
print("\nSplit-half control, alternate WAKE epochs: slope of one half on the other %.3f (95%% CI %.3f to %.3f), "
      "R = %.3f, n = %d (%d cells with a zero rate in a half left out)" % (b, lo, hi, r_h, n, (~kh).sum()))
print("Reliability of the log10 WAKE rate (Spearman-Brown) %.3f; nonREM-on-WAKE slope corrected for attenuation %.3f "
      "(%.3f to %.3f, the reliability's own uncertainty not counted)" % (rel, *(v / rel for v in fits["(a) log10"][:3])))

**Result: reproduced.** Under reading (a), log10 rates, R = 0.862 and the slope of nonREM on WAKE is 0.757 (95% CI 0.727 to 0.788), inside the widened interval 0.594 to 0.781 and below 1. In Hz the slope is 0.583, outside the interval. The reverse regression, WAKE on nonREM, gives 0.982 (0.942 to 1.022), so part of the slope below 1 can come from regression to the mean. The split-half control puts that part at a few percent: the slope of one half of a cell's WAKE epochs on the other is 0.936, the reliability of the log10 WAKE rate is 0.961, and the nonREM-on-WAKE slope corrected for attenuation is 0.788 (0.756 to 0.821), still below 1. The criterion does not use these checks.

## fig2-burst-index-by-state

> The burst index (fraction of spikes with inter-spike intervals under 15 ms) is highest in
> nonREM (median 11.8%), lowest in WAKE (4.0%), with REM (5.5%) slightly above WAKE
> (Kruskal-Wallis, p < 0.05).

In [ ]:
bi = []
for s in sessions:
    for u in s.pe:
        row = dict(subject=s.subject, session=s.session_id, unit=u)
        for state in ("WAKE", "NREM", "REM"):
            m = s.state == state
            for rd in ("a", "b"):
                row[f"{state}_{rd}"] = ss.burst_index(s.spikes[u], s.starts[m], s.stops[m], rd)
        bi.append(row)
bi = pd.DataFrame(bi)
for rd in ("a", "b"):
    cols = [f"{x}_{rd}" for x in ("NREM", "WAKE", "REM")]
    med = 100 * bi[cols].median()
    h, p = st.kruskal(*[bi[c].dropna() for c in cols])
    print("Reading (%s): medians %% NREM %.2f, WAKE %.2f, REM %.2f; n %s; Kruskal-Wallis H = %.1f, p = %.2g" % (
        rd, *med.values, bi[cols].notna().sum().to_dict(), h, p))
    full = bi[cols].dropna()
    print("   Friedman test on the %d cells with all three values (paired): p = %.2g" % (
        len(full), st.friedmanchisquare(*[full[c] for c in cols]).pvalue))
    print("   within tolerance:", [abs(m_ - v) <= 0.1 * v for m_, v in zip(med.values, (11.8, 4.0, 5.5))],
          "| order NREM > REM > WAKE:", med.iloc[0] > med.iloc[2] > med.iloc[1])
    pair = st.wilcoxon(bi[f"REM_{rd}"], bi[f"WAKE_{rd}"], nan_policy="omit")
    print("   REM against WAKE, paired Wilcoxon p = %.2g" % pair.pvalue)
print("Cells left out for fewer than 10 intervals:", bi[[f"{x}_a" for x in ("NREM", "WAKE", "REM")]].isna().sum().to_dict())
print("\nPer subject, median burst index % (reading a):")
print((100 * bi.groupby("subject")[["NREM_a", "WAKE_a", "REM_a"]].median()).round(2).to_string())

**Result: partially reproduced.** The order nonREM > REM > WAKE holds under both readings (Kruskal-Wallis p = 5.7e-36 under (a); the paired Friedman test gives p = 3.9e-151), and REM is above WAKE (paired Wilcoxon p = 1e-32). The medians match only in part: reading (a) gives 6.35, 2.64 and 3.65%, none within tolerance; reading (b) gives 11.67% in nonREM (within tolerance), 4.95% in WAKE and 6.71% in REM (both above).

## fig2c-acg-peaks

> The average spike autocorrelogram of pE cells peaks near 8 ms in nonREM, with a smaller peak near
> 125 ms (theta) in REM and WAKE.

In [ ]:
# The helper on a synthetic train with a spike every 10 ms: every pair must fall in the 10, 20, ...,
# 290 ms bins (lags are rounded to 1 ns before binning, so float error cannot move a pair across an edge).
h_syn, _ = ss.autocorrelogram(np.arange(0, 10, 0.01), [0.0], [10.0])
print("Synthetic 10 ms train, non-empty bins (ms):", np.nonzero(h_syn)[0].tolist())
assert np.nonzero(h_syn)[0].tolist() == list(range(10, 300, 10))
acg = {state: [] for state in ("NREM", "WAKE", "REM")}
acg_sess = {state: [] for state in acg}
for s in sessions:
    for u in s.pe:
        for state in acg:
            m = s.state == state
            h, n = ss.autocorrelogram(s.spikes[u], s.starts[m], s.stops[m])
            if n >= 100:
                acg[state].append(h / n)
                acg_sess[state].append(s.session_id)
lags = np.arange(300) + 0.5
res = {}
for state, hs in acg.items():
    mean = np.mean(hs, axis=0)
    early = np.argmax(mean[2:50]) + 2
    sm = np.convolve(mean, np.ones(21) / 21, mode="same")
    w = (lags >= 60) & (lags <= 250)
    k = np.where(w)[0][np.argmax(sm[w])]
    res[state] = (lags[early], lags[k], len(hs), k)
    print("%s: cells %d; peak 2-50 ms at %.1f ms; smoothed maximum 60-250 ms at %.1f ms (%s the window edges)" % (
        state, len(hs), lags[early], lags[k], "inside" if 63 <= lags[k] <= 247 else "at"))
for state, hs in acg.items():
    mean = np.mean(hs, axis=0)
    hp = np.convolve(mean, np.ones(21) / 21, mode="same") / np.convolve(mean, np.ones(101) / 101, mode="same")
    w = (lags >= 80) & (lags <= 200)
    print("Variant, %s: 21 ms over 101 ms moving average, maximum 80-200 ms at %.1f ms (ratio %.3f)" % (
        state, lags[w][np.argmax(hp[w])], hp[w].max()))
print("\nNREM peak within 7-9 ms:", 7 <= res["NREM"][0] <= 9,
      "| theta peak within 112.5-137.5 ms in REM:", 112.5 <= res["REM"][1] <= 137.5,
      "and WAKE:", 112.5 <= res["WAKE"][1] <= 137.5)
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
for state, c in (("WAKE", "k"), ("NREM", "tab:blue"), ("REM", "tab:red")):
    mean = np.mean(acg[state], axis=0)
    ax[0].plot(lags[:50], mean[:50], color=c, label=state)
    ax[1].plot(lags[30:], np.convolve(mean, np.ones(21) / 21, mode="same")[30:] / mean[30:].mean(), color=c)
ax[0].set_xlabel("lag (ms)"); ax[0].set_ylabel("count per spike per ms"); ax[0].legend()
ax[1].set_xlabel("lag (ms)"); ax[1].set_ylabel("smoothed, relative to mean 30-300 ms")
plt.show()
for state in acg:
    mean = np.mean(acg[state], axis=0)
    print("%s mean ACG range 0-50 ms %.4f to %.4f; smoothed relative 30-300 ms %.3f to %.3f" % (
        state, mean[:50].min(), mean[:50].max(),
        *(lambda v: (v.min(), v.max()))((np.convolve(mean, np.ones(21) / 21, mode="same")[30:] / mean[30:].mean())[10:-10])))
print("\nPer subject, NREM early peak (ms) and theta-window maximum in REM and WAKE (ms):")
for subj in sorted(tab.subject.unique()):
    out = []
    for state in ("NREM", "REM", "WAKE"):
        hs = []
        for s in sessions:
            if s.subject != subj:
                continue
            for u in s.pe:
                m = s.state == state
                h, n = ss.autocorrelogram(s.spikes[u], s.starts[m], s.stops[m])
                if n >= 100:
                    hs.append(h / n)
        if not hs:
            out.append(None); continue
        mean = np.mean(hs, axis=0)
        sm = np.convolve(mean, np.ones(21) / 21, mode="same")
        w = (lags >= 60) & (lags <= 250)
        out.append(lags[np.argmax(mean[2:50]) + 2] if state == "NREM" else lags[np.where(w)[0][np.argmax(sm[w])]])
    print("  %s: NREM %s, REM %s, WAKE %s" % (subj, *out))

**Result: partially reproduced.** The nonREM autocorrelogram peaks at 8.5 ms, within 7 to 9 ms. No theta peak is found by the criterion: the smoothed REM and WAKE autocorrelograms are highest at the 60 ms edge of the 60 to 250 ms window, as they decline from short lags. In the variant that divides by a 101 ms moving average, WAKE has a maximum at 118.5 ms (ratio 1.016) and REM at 142.5 ms (1.017); this does not change the outcome. As the smoothed autocorrelograms fall from short lags, the criterion's window maximum lands at its lower edge unless the theta peak rises above that decline.

## fig2f-rate-group-correlations

> In nonREM, pairs of fast-firing pE cells are positively correlated, while pairs of one slow and
> one fast cell are negatively correlated.

In [ ]:
# sextiles of the whole-session rate, reading (b)
whole = {s.path: np.array([len(s.spikes[u]) / (s.stops[-1] - s.starts[0]) for u in s.pe]) for s in sessions}
edges_b = np.quantile(np.concatenate(list(whole.values())), np.linspace(0, 1, 7))
groups_b = {p: np.searchsorted(edges_b[1:-1], r, side="right") + 1 for p, r in whole.items()}
pairs = []
n_pairs_all = 0
for s in sessions:
    m = s.state == "NREM"
    binned = []
    for a, b in zip(s.starts[m], s.stops[m]):
        e = np.arange(a, b, 0.1)
        if len(e) > 1:
            binned.append(np.array([np.diff(np.searchsorted(s.spikes[u], e)) for u in s.pe]))
    X = np.concatenate(binned, axis=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        C = np.corrcoef(X)
    n_pairs_all += len(s.pe) * (len(s.pe) - 1) // 2
    ga, gb = sext[s.path][1], groups_b[s.path]
    iu = np.triu_indices(len(s.pe), 1)
    for i, j in zip(*iu):
        if np.isfinite(C[i, j]):
            pairs.append(dict(subject=s.subject, session=s.session_id, r=C[i, j],
                              a=tuple(sorted((ga[i], ga[j]))), b=tuple(sorted((gb[i], gb[j])))))
pairs = pd.DataFrame(pairs)
print("pairs with a defined correlation %d of %d (cells without Non-REM spikes have none)" % (len(pairs), n_pairs_all))
print("pairs:", len(pairs), "| overall mean r %.4f, fraction negative %.3f" % (pairs.r.mean(), np.mean(pairs.r < 0)))
for rd in ("a", "b"):
    g = pairs.groupby(rd).r.agg(["mean", "size"])
    mat = np.full((6, 6), np.nan)
    for (i, j), v in g["mean"].items():
        mat[i - 1, j - 1] = mat[j - 1, i - 1] = v
    print("\nReading (%s), mean r by group pair (rows and columns: groups 1 to 6):" % rd)
    print(pd.DataFrame(mat, index=range(1, 7), columns=range(1, 7)).round(4).to_string())
    r16 = pairs[pairs[rd] == (1, 6)].r
    t = st.ttest_1samp(r16, 0)
    top = g["mean"].idxmax()
    print("(6,6) mean %.4f, largest pair is %s; (1,6) mean %.4f, n %d, t = %.2f, p = %.2g" % (
        g["mean"][(6, 6)], top, r16.mean(), len(r16), t.statistic, t.pvalue))
print("\nPer subject, reading (a): mean r of (6,6) and (1,6) pairs")
print(pairs.groupby(["subject", "a"]).r.mean().unstack()[[(6, 6), (1, 6)]].round(4).to_string())

**Result: partially reproduced.** Under both readings the mean correlation of (6,6) pairs is positive and the largest of the 21 combinations (0.0424 under reading (a)), and correlations rise with the rates of both cells. No pair group is negatively correlated: (1,6) pairs have a mean of 0.0089 (t = 10.85, p = 8.7e-26, above zero), and 22.8% of all pairs are negative. Correlations of raw 100 ms counts may differ from the paper's measure, whose definition is in the supplement that was not available.

## ma-statistics

> Microarousals have a median duration of 15.9 plus or minus 3.3 s, occur 0.32 plus or minus 0.12
> times per minute and occupy 9.1 plus or minus 3.5% of SLEEP (790 in total).

In [ ]:
rows = []
all_ma = []
ep_time = ep_ma_n = 0
for s in sessions:
    dur = s.stops - s.starts
    all_ma.append(dur[s.state == "MA"])
    periods = ss.sleep_periods(s.starts, s.stops, s.labels)
    n = T = tma = 0
    durs_in = []
    for p_ in periods:
        rr = np.arange(p_["first"], p_["last"] + 1)
        ma = rr[s.state[rr] == "MA"]
        n += len(ma); tma += dur[ma].sum(); T += p_["stop"] - p_["start"]; durs_in += list(dur[ma])
    for a, b in ss.nrem_episodes(s.state):
        ep_time += s.stops[b] - s.starts[a]
        ep_ma_n += int(np.sum(s.state[a:b + 1] == "MA"))
    rows.append(dict(subject=s.subject, session=s.session_id, periods=len(periods), MA_in_sleep=n,
                     sleep_min=T / 60, freq=n / (T / 60) if T else np.nan, frac=100 * tma / T if T else np.nan,
                     durs=durs_in))
ma = pd.DataFrame(rows)
in_durs = np.concatenate([np.array(x, float) for x in ma.durs])
all_ma = np.concatenate(all_ma)
print(ma.drop(columns="durs").round(3).to_string())
a = (np.median(in_durs), ma.freq.mean(), ma.frac.mean())
b = (np.median(all_ma), ep_ma_n / (ep_time / 60), 100 * all_ma.sum() / ep_time)
for name, v in (("(a) inside SLEEP, per-session means", a), ("(b) all MAs, pooled, per nonREM episode time", b)):
    ok = [abs(v[0] - 15.9) <= 3.3, abs(v[1] - 0.32) <= 0.12, abs(v[2] - 9.1) <= 3.5]
    print("%s: median duration %.1f s, frequency %.3f per min, fraction %.2f%%; within tolerance %s" % (name, *v, ok))
print("SD over sessions (a): frequency %.3f, fraction %.2f" % (ma.freq.std(), ma.frac.std()))
print("MAs: %d in all, %d inside SLEEP periods (paper: 790 in 27 sessions)" % (len(all_ma), len(in_durs)))
print("MA durations (s): min %.0f, max %.0f; the state table has 1 s resolution" % (all_ma.min(), all_ma.max()))
print("Awake epochs under 40 s not between two Non-REM epochs (left as WAKE):",
      sum(int(np.sum((s.labels == "Awake") & (s.stops - s.starts < 40) & (s.state == "WAKE"))) for s in sessions))

In [ ]:
# Awake epochs under 40 s by their neighbours in the state table, and MA counts under looser rules,
# counted inside SLEEP periods (per minute of SLEEP and percent of SLEEP time, per-session means as in
# reading (a)). The first rule is the one used throughout.
def neighbour(s, j):
    return s.labels[j] if 0 <= j < len(s.labels) else "edge"

short_n = collections.Counter()
short_d = collections.defaultdict(list)
for s in sessions:
    dur = s.stops - s.starts
    for i in np.where((s.labels == "Awake") & (dur < ss.MA_MAX))[0]:
        key = (neighbour(s, i - 1), neighbour(s, i + 1))
        short_n[key] += 1
        short_d[key].append(dur[i])
print("Awake epochs under 40 s by previous and next label: count, median duration")
for key, v in short_n.most_common():
    print("  %s, Awake, %s: %d, median %.0f s" % (key[0], key[1], v, np.median(short_d[key])))

both = lambda s, i: neighbour(s, i - 1) == "Non-REM" and neighbour(s, i + 1) == "Non-REM"
one = lambda s, i: "Non-REM" in (neighbour(s, i - 1), neighbour(s, i + 1))
rules = {"Non-REM on both sides, under 40 s (used here)": lambda s, i, d: d < 40 and both(s, i),
         "Non-REM on at least one side, under 40 s": lambda s, i, d: d < 40 and one(s, i),
         "any neighbours, under 40 s": lambda s, i, d: d < 40,
         "Non-REM on both sides, under 60 s": lambda s, i, d: d < 60 and both(s, i),
         "Non-REM on at least one side, under 60 s": lambda s, i, d: d < 60 and one(s, i),
         "every Awake epoch inside SLEEP (all are 120 s or shorter)": lambda s, i, d: True}
print("\nMAs inside SLEEP under each rule (paper: 0.32 per min, 9.1% of SLEEP):")
for name, rule in rules.items():
    freq, frac, n_all = [], [], 0
    for s in sessions:
        dur = s.stops - s.starts
        n = T = tma = 0
        for p_ in ss.sleep_periods(s.starts, s.stops, s.labels):
            for i in range(p_["first"], p_["last"] + 1):
                if s.labels[i] == "Awake" and rule(s, i, dur[i]):
                    n += 1
                    tma += dur[i]
            T += p_["stop"] - p_["start"]
        if T:
            freq.append(n / (T / 60)); frac.append(100 * tma / T); n_all += n
    print("  %s: %d, %.3f per min, %.2f%% of SLEEP" % (name, n_all, np.mean(freq), np.mean(frac)))

**Result: not reproduced.** The median MA duration, 19.0 s, is within the paper's spread, but MAs are rarer: 0.096 per min of SLEEP and 3.22% of SLEEP time under reading (a), 0.127 per min and 4.23% under (b), against 0.32 plus or minus 0.12 per min and 9.1 plus or minus 3.5%. The 21 sessions give 391 MAs (262 inside SLEEP periods), against 790 in 27 sessions. MAs are derived here from the Awake label; 345 short Awake epochs that are not between two Non-REM epochs are left as WAKE (186 of them between REM and Non-REM). Looser rules raise the counts but no rule reaches both of the paper's values: Non-REM on at least one side and under 60 s gives 0.177 per min and 7.63% of SLEEP, and counting every Awake epoch inside SLEEP as an MA gives 0.245 per min and 13.66%.

## ma-fraction-over-nrem-episodes

> The fraction of time occupied by MAs decreases over the course of nonREM episodes (R = -0.85,
> p < 1e-10).

In [ ]:
fr, fr_sess = [], []
for s in sessions:
    for a, b in ss.nrem_episodes(s.state):
        rr = np.arange(a, b + 1)
        mas = [(s.starts[i], s.stops[i]) for i in rr if s.state[i] == "MA"]
        if not mas:
            continue
        bins = ss.binned_intervals(s.starts[a], s.stops[b], 10, mas)
        width = (s.stops[b] - s.starts[a]) / 10
        fr_sess.append(s.session_id)
        fr.append([np.sum(e - st_) / width for st_, e in bins])
fr = np.array(fr)
x = np.tile(np.arange(10) + 0.5, len(fr))
ra = st.pearsonr(np.arange(10) + 0.5, fr.mean(axis=0))
rb = st.pearsonr(x, fr.ravel())
print("Episodes with at least one MA:", len(fr))
print("Mean MA fraction per bin:", np.round(fr.mean(axis=0), 4))
print("(a) bin means: R = %.3f, p = %.2g | (b) pooled: R = %.3f, p = %.2g" % (ra[0], ra[1], rb[0], rb[1]))
print("Within -0.935 to -0.765:", {"a": -0.935 <= ra[0] <= -0.765, "b": -0.935 <= rb[0] <= -0.765})
# the first and last bins start and end in a packet by construction
print("Variant without the first and last bins: (a) R = %.3f, p = %.2g" % st.pearsonr(np.arange(1, 9) + 0.5, fr.mean(axis=0)[1:-1]))

**Result: partially reproduced.** Over 243 episodes with an MA, the MA fraction pooled over episode bins decreases with time (R = -0.087, p = 1.7e-05), but neither reading reaches R = -0.85 or p < 1e-10: the bin means give R = -0.451 (p = 0.19). The mean fraction rises over the first half of the episode (0.067 to 0.157) and falls in the second (to 0.010); the first and last bins start and end in a packet by construction.

## packet-duration

> The duration of nonREM packets spans two orders of magnitude, with a median of 111 s.

In [ ]:
pa, pb = [], []
for s in sessions:
    dur = s.stops - s.starts
    pa.append(dur[s.state == "NREM"])
    for p_ in ss.sleep_periods(s.starts, s.stops, s.labels):
        rr = np.arange(p_["first"], p_["last"] + 1)
        pb.append(dur[rr][s.state[rr] == "NREM"])
pa, pb = np.concatenate(pa), np.concatenate(pb)
for name, v in (("(a) all", pa), ("(b) inside SLEEP", pb)):
    print("%s: n %d, median %.1f s, 5th to 95th pct %.0f to %.0f s, min %.0f, max %.0f; within 99.9-122.1: %s" % (
        name, len(v), np.median(v), *np.percentile(v, [5, 95]), v.min(), v.max(), 99.9 <= np.median(v) <= 122.1))
print("Per subject median (a):", {subj: float(np.median(np.concatenate([(s.stops - s.starts)[s.state == "NREM"]
      for s in sessions if s.subject == subj]))) for subj in sorted(tab.subject.unique())})

**Result: reproduced.** Over all 1069 Non-REM epochs the median duration is 108 s (within 99.9 to 122.1), with a 5th to 95th percentile of 12 to 484 s and a range of 6 to 1199 s, two orders of magnitude. Inside SLEEP periods the median is 128 s. Per subject the median ranges from 76 to 219 s.

## wake-sleep-cycles

> The dataset holds 54 WAKE-SLEEP episodes, an average of 2.0 per session, and sessions without one
> were discarded.

In [ ]:
cyc = {s.path: ss.cycles(s) for s in sessions}
cyc_open = {s.path: ss.cycles(s, require_wake_end=False) for s in sessions}
nc = pd.Series({s.session_id: len(cyc[s.path]) for s in sessions})
nco = pd.Series({s.session_id: len(cyc_open[s.path]) for s in sessions})
print(pd.DataFrame({"cycles": nc, "cycles incl. SLEEP ended by the recording": nco}).to_string())
print("Total %d, mean per session %.2f, sessions with none: %d | variant: total %d, mean %.2f, none: %d" % (
    nc.sum(), nc.mean(), (nc == 0).sum(), nco.sum(), nco.mean(), (nco == 0).sum()))
print("Within 1.8-2.2:", 1.8 <= nc.mean() <= 2.2)
pw = [s.stops[p_["pre"]] - s.starts[p_["pre"]] for s in sessions for p_ in cyc[s.path]]
sl = [(p_["stop"] - p_["start"]) / 60 for s in sessions for p_ in cyc[s.path]]
print("Pre-sleep WAKE (min): %.1f to %.1f; SLEEP duration (min): median %.1f, range %.1f to %.1f" % (
    min(pw) / 60, max(pw) / 60, np.median(sl), min(sl), max(sl)))
print("\nSessions without a cycle: each sleep segment between Awake epochs over 120 s (leading and trailing Awake"
      " trimmed), the Awake before it, and why it is not a cycle; segments under 1 min not listed")
for s in sessions:
    if cyc[s.path]:
        continue
    dur = s.stops - s.starts
    long_wake = np.where((s.labels == "Awake") & (dur > ss.SLEEP_BREAK))[0]
    cuts = [-1] + list(long_wake) + [len(s.labels)]
    print("  %s: %.0f min of states, %d Awake epochs over 120 s" % (s.session_id, (s.stops[-1] - s.starts[0]) / 60, len(long_wake)))
    for lo_, hi_ in zip(cuts[:-1], cuts[1:]):
        rr = [r for r in range(lo_ + 1, hi_) if s.labels[r] != "Awake"]
        if not rr or s.stops[rr[-1]] - s.starts[rr[0]] < 60:
            continue
        seg = (s.stops[rr[-1]] - s.starts[rr[0]]) / 60
        pre_ = dur[lo_] / 60 if lo_ >= 0 else None
        why = ([] if seg >= 20 else ["shorter than 20 min"]) + ([] if hi_ < len(s.labels) else ["ended by the recording"]) + \
              (["no Awake before"] if pre_ is None else [] if pre_ >= 7 else ["Awake before under 7 min"])
        print("    %.1f to %.1f min: %.1f min, Awake before %s; %s" % (
            s.starts[rr[0]] / 60, s.stops[rr[-1]] / 60, seg, "none" if pre_ is None else "%.1f min" % pre_,
            ", ".join(why) or "a cycle"))

**Result: not reproduced.** The definitions give 30 WAKE-SLEEP cycles in 21 sessions, a mean of 1.43 per session (1.8 to 2.2 expected), and 3 sessions have none, although the paper kept only sessions with a full cycle. Accepting SLEEP ended by the end of the recording gives 34 cycles, a mean of 1.62, and one session without a cycle. The state table may not be the scoring the paper used (see the data check), so this may reflect the scoring as much as the definitions. In the three sessions without a cycle, the long sleep segments fail because the Awake before them lasts under 7 min (3.6 and 5.0 min) or because the recording ends them (BWRat18-020513 and BWRat20-101013).

## fig3b-population-decline

> The mean firing rate of pE cells declines over the course of SLEEP, restricted to nonREM
> (R = -0.11, p < 1e-3).

In [ ]:
def group_mean(rmat, units_mask):
    """Mean over the cells in the mask, cells with no spikes over the whole interval dropped,
    divided by its mean over the points; None when undefined."""
    sub = rmat[units_mask]
    sub = sub[np.nansum(sub, axis=1) > 0]
    if len(sub) == 0:
        return None
    m = np.nanmean(sub, axis=0)
    mu = np.nanmean(m)
    return m / mu if mu > 0 else None

def sleep_points(nbins=10, min_nrem=10.0):
    """Per cycle and reading: x, rate matrix (pE x points), session path."""
    out = {"a": [], "b": []}
    for s in sessions:
        for p_ in cyc[s.path]:
            rr = np.arange(p_["first"], p_["last"] + 1)
            pk = rr[s.state[rr] == "NREM"]
            T = p_["stop"] - p_["start"]
            xa = ((s.starts[pk] + s.stops[pk]) / 2 - p_["start"]) / T
            ra = np.array([[ss.counts(s.spikes[u], s.starts[i], s.stops[i]) / (s.stops[i] - s.starts[i]) for i in pk] for u in s.pe])
            out["a"].append((xa, ra, s))
            bins = ss.binned_intervals(p_["start"], p_["stop"], nbins, list(zip(s.starts[pk], s.stops[pk])))
            rb, dur = ss.bin_rates(s, s.pe, bins)
            ok = dur >= min_nrem
            out["b"].append(((np.arange(nbins) + 0.5)[ok] / nbins, rb[:, ok], s))
    return out

pts = sleep_points()

def pooled_r(pts_r, groups=None):
    """Pooled Pearson R of normalized group means against x. groups None = all pE cells."""
    xs, ys = [], []
    for x, rmat, s in pts_r:
        mask = np.ones(len(s.pe), bool) if groups is None else np.isin(sext[s.path][1], groups)
        v = group_mean(rmat, mask)
        if v is not None:
            xs.append(x); ys.append(v)
    x, y = np.concatenate(xs), np.concatenate(ys)
    ok = np.isfinite(y)
    r, p = st.pearsonr(x[ok], y[ok])
    return r, p, ok.sum()

for rd in ("a", "b"):
    r, p, n = pooled_r(pts[rd])
    print("Reading (%s): R = %.3f, p = %.2g, points %d; within -0.121 to -0.099: %s" % (rd, r, p, n, -0.121 <= r <= -0.099))
print("\nPer subject, reading (a):")
for subj in sorted(tab.subject.unique()):
    sub = [t_ for t_ in pts["a"] if t_[2].subject == subj]
    if sub:
        r, p, n = pooled_r(sub)
        print("  %s: R = %.3f, p = %.2g, points %d, cycles %d" % (subj, r, p, n, len(sub)))

**Result: partially reproduced.** The population rate declines over SLEEP under both readings, more steeply than reported: R = -0.267 (p = 4.3e-06, 287 packets) per packet and R = -0.285 (p = 7.6e-07) in bins, against R = -0.11. The per-packet R is negative in 8 of 9 subjects with a cycle; two of them have p < 0.05 and one p = 0.05.

## fig3-first-last-packet

> The mean pE rate is lower in the last than in the first nonREM packet of SLEEP (1.04 plus or minus
> 1.20 against 0.88 plus or minus 0.91 Hz; Wilcoxon p < 0.02).

In [ ]:
fl = []
for s in sessions:
    for c, p_ in enumerate(cyc[s.path]):
        rr = np.arange(p_["first"], p_["last"] + 1)
        pk = rr[s.state[rr] == "NREM"]
        r1 = ss.rates(s.spikes, s.pe, s.starts[pk[0]], s.stops[pk[0]])
        r2 = ss.rates(s.spikes, s.pe, s.starts[pk[-1]], s.stops[pk[-1]])
        for k, u in enumerate(s.pe):
            fl.append(dict(subject=s.subject, session=s.session_id, cycle=c, unit=u, first=r1[k], last=r2[k],
                           sextile=sext[s.path][1][k], first_dur=s.stops[pk[0]] - s.starts[pk[0]],
                           last_dur=s.stops[pk[-1]] - s.starts[pk[-1]],
                           first_half=ss.rates(s.spikes, [u], s.starts[pk[0]], (s.starts[pk[0]] + s.stops[pk[0]]) / 2)[0],
                           second_half=ss.rates(s.spikes, [u], (s.starts[pk[0]] + s.stops[pk[0]]) / 2, s.stops[pk[0]])[0]))
fl = pd.DataFrame(fl)
w = st.wilcoxon(fl["first"], fl["last"])
print("cell-cycle pairs %d (cycles %d); first %.4f +- %.3f Hz, last %.4f +- %.3f Hz; Wilcoxon p = %.2g; last lower: %s" % (
    len(fl), fl.groupby(["session", "cycle"]).ngroups, fl["first"].mean(), fl["first"].std(), fl["last"].mean(),
    fl["last"].std(), w.pvalue, fl["last"].mean() < fl["first"].mean()))
print("Within tolerance: first", abs(fl["first"].mean() - 1.04) <= 0.104, "last", abs(fl["last"].mean() - 0.88) <= 0.088)
print("Medians: first %.3f, last %.3f; first packet duration median %.0f s, last %.0f s" % (
    fl["first"].median(), fl["last"].median(), fl.groupby(["session", "cycle"]).first_dur.first().median(),
    fl.groupby(["session", "cycle"]).last_dur.first().median()))
print("\nPer subject:")
for subj, g in fl.groupby("subject"):
    print("  %s: first %.3f, last %.3f, n %d, Wilcoxon p = %.2g" % (subj, g["first"].mean(), g["last"].mean(), len(g),
                                                                    st.wilcoxon(g["first"], g["last"]).pvalue))
pcyc = fl.groupby(["session", "cycle"])[["first", "last"]].mean()
print("\nOne value per cycle (mean over its pE cells): first %.3f, last %.3f Hz; last lower in %d of %d cycles, "
      "Wilcoxon over cycles p = %.2g" % (pcyc["first"].mean(), pcyc["last"].mean(), (pcyc["last"] < pcyc["first"]).sum(),
                                        len(pcyc), st.wilcoxon(pcyc["first"], pcyc["last"]).pvalue))

**Result: reproduced.** Over 1070 cell and cycle pairs the mean rate is 1.0915 Hz in the first packet and 0.9679 Hz in the last (both within tolerance, the last at its edge, 0.968), and the decrease is significant (Wilcoxon p = 3.9e-06). It is significant in 3 of 9 subjects alone. With one value per cycle the last packet is lower in 22 of 30 cycles (Wilcoxon p = 0.015).

## fig3d-first-last-slope

> Comparing the same neurons in the first and last nonREM packets of SLEEP, the regression slope
> departs from 1 (95% confidence interval 0.827 to 0.880).

In [ ]:
k2 = (fl["first"] > 0) & (fl["last"] > 0)
fits = {"(a) log10, last on first": ols(np.log10(fl["first"][k2]), np.log10(fl["last"][k2])),
        "(b) Hz, last on first": ols(fl["first"], fl["last"]),
        "(a) reverse, first on last": ols(np.log10(fl["last"][k2]), np.log10(fl["first"][k2]))}
k3 = (fl.first_half > 0) & (fl.second_half > 0)
fits["check: second half on first half of the first packet, log10"] = ols(np.log10(fl.first_half[k3]), np.log10(fl.second_half[k3]))
for k, (b, lo, hi, r, p, n) in fits.items():
    print("%s: slope %.3f (95%% CI %.3f to %.3f), R = %.3f, n = %d" % (k, b, lo, hi, r, n))
print("Pairs dropped for a zero rate (a): %d" % (~k2).sum())
for k in ("(a) log10, last on first", "(b) Hz, last on first"):
    b, lo, hi = fits[k][:3]
    print(k, "| slope within 0.744-0.968:", 0.744 <= b <= 0.968, "| CI excludes 1:", hi < 1)
r_h = fits["check: second half on first half of the first packet, log10"][3]
rel = 2 * r_h / (1 + r_h)
print("Reliability of the log10 first-packet rate (Spearman-Brown, from its halves) %.3f; last-on-first slope corrected "
      "for attenuation %.3f (%.3f to %.3f, the reliability's own uncertainty not counted)" % (
          rel, *(v / rel for v in fits["(a) log10, last on first"][:3])))

**Result: reproduced.** The slope of last on first packet rates is 0.801 (95% CI 0.768 to 0.834) on log10 rates, inside the widened interval and below 1; in Hz it is 0.723. The checks do not separate this from regression to the mean: the reverse regression is also below 1 (0.864, CI 0.829 to 0.900), and the second half of the first packet regressed on its first half gives 0.855 (0.829 to 0.881), as low as the first-to-last slope. Corrected for the noise in the first-packet rate (reliability 0.946 from its halves), the slope is 0.846 (0.812 to 0.881), still below 1, so measurement noise does not account for all of it. The outcome follows the criterion.

## fig3e-wake-before-after

> pE rates are lower in the 5 min of WAKE after SLEEP than in the 5 min before (p < 1e-6,
> Wilcoxon), with a decrease in the top three sextiles and an increase in the lowest.

In [ ]:
ba = []
for s in sessions:
    for c, p_ in enumerate(cyc[s.path]):
        pre, post = p_["pre"], p_["post"]
        r1 = ss.rates(s.spikes, s.pe, s.stops[pre] - 300, s.stops[pre])
        r2 = ss.rates(s.spikes, s.pe, s.starts[post], min(s.starts[post] + 300, s.stops[post]))
        for k, u in enumerate(s.pe):
            ba.append(dict(subject=s.subject, session=s.session_id, cycle=c, unit=u, pre=r1[k], post=r2[k],
                           sextile=sext[s.path][1][k], post_len=min(300, s.stops[post] - s.starts[post])))
ba = pd.DataFrame(ba)
print("Post-sleep WAKE windows shorter than 300 s: %d of %d cycles" % (
    (ba.groupby(["session", "cycle"]).post_len.first() < 300).sum(), ba.groupby(["session", "cycle"]).ngroups))
for rd, d in (("(a) cell-cycle pairs", ba),
              ("(b) per-cell means", ba.groupby(["session", "unit"]).agg(pre=("pre", "mean"), post=("post", "mean"),
                                                                         sextile=("sextile", "first")).reset_index())):
    w = st.wilcoxon(d.pre, d.post)
    print("\n%s: n %d, mean pre %.3f, post %.3f Hz, median change %.4f, Wilcoxon p = %.2g" % (
        rd, len(d), d.pre.mean(), d.post.mean(), (d.post - d.pre).median(), w.pvalue))
    for g in range(1, 7):
        x = d[d.sextile == g]
        alt = "greater" if g == 1 else "less"
        pv = st.wilcoxon(x.post, x.pre, alternative=alt).pvalue
        print("  sextile %d: n %d, mean pre %.3f, post %.3f, mean change %+.3f, one-tailed (%s) p = %.2g" % (
            g, len(x), x.pre.mean(), x.post.mean(), (x.post - x.pre).mean(), "increase" if g == 1 else "decrease", pv))
print("\nPer subject, (a): mean change post - pre (Hz) for all pE, sextile 1 and sextile 6")
print(ba.assign(ch=ba.post - ba.pre).groupby("subject").apply(
    lambda g: pd.Series({"all": g.ch.mean(), "s1": g.ch[g.sextile == 1].mean(), "s6": g.ch[g.sextile == 6].mean(), "n": len(g)}),
    include_groups=False).round(3).to_string())
# Variant with windows of equal length: the pre-sleep window is cut to the length of the post-sleep one.
ba2 = []
for s in sessions:
    for c, p_ in enumerate(cyc[s.path]):
        pre, post = p_["pre"], p_["post"]
        L = min(300, s.stops[post] - s.starts[post])
        r1 = ss.rates(s.spikes, s.pe, s.stops[pre] - L, s.stops[pre])
        r2 = ss.rates(s.spikes, s.pe, s.starts[post], s.starts[post] + L)
        for k, u in enumerate(s.pe):
            ba2.append(dict(session=s.session_id, cycle=c, unit=u, pre=r1[k], post=r2[k], sextile=sext[s.path][1][k], L=L))
ba2 = pd.DataFrame(ba2)

def before_after(d, label):
    out = []
    for g in range(1, 7):
        x = d[d.sextile == g]
        out.append("s%d %+.3f (p %.2g)" % (g, (x.post - x.pre).mean(),
                                           st.wilcoxon(x.post, x.pre, alternative="greater" if g == 1 else "less").pvalue))
    print("%s: n %d, cycles %d, mean pre %.3f, post %.3f Hz, p = %.2g\n   %s" % (
        label, len(d), d.groupby(["session", "cycle"]).ngroups, d.pre.mean(), d.post.mean(), st.wilcoxon(d.pre, d.post).pvalue,
        ", ".join(out)))

print("\nVariants (mean change per sextile in Hz, one-tailed p):")
before_after(ba2, "pre window cut to the length of the post window")
print("   window lengths (s): median %.0f, range %.0f to %.0f" % (
    ba2.groupby(["session", "cycle"]).L.first().median(), ba2.L.min(), ba2.L.max()))
before_after(ba[ba.post_len >= 300], "only cycles with 300 s of WAKE after SLEEP")
print("\nOne value per cycle (mean change over its cells), and per session; Wilcoxon over these (two-sided):")
chg = ba.assign(ch=ba.post - ba.pre)
for label, g in [("all pE", None)] + [("sextile %d" % g, g) for g in range(1, 7)]:
    x = chg if g is None else chg[chg.sextile == g]
    cm, sm = x.groupby(["session", "cycle"]).ch.mean(), x.groupby("session").ch.mean()
    print("  %s: per cycle mean %+.3f, lower in %d of %d, p = %.2g | per session lower in %d of %d, p = %.2g" % (
        label, cm.mean(), (cm < 0).sum(), len(cm), st.wilcoxon(cm).pvalue, (sm < 0).sum(), len(sm), st.wilcoxon(sm).pvalue))

**Result: partially reproduced.** pE rates are lower in the 5 min of WAKE after SLEEP (1.180 against 1.341 Hz, Wilcoxon p = 4.1e-09, reading (a)), and sextiles 4, 5 and 6 decrease (one-tailed p = 0.03, 0.00063 and 2.4e-05). The slowest sextile does not increase (mean change -0.002 Hz, p = 0.3); sextiles 2 and 3 decrease as well. Reading (b) gives the same pattern. In 23 of 30 cycles the WAKE after SLEEP is shorter than 300 s; with the pre-sleep window cut to the same length the result is the same (p = 1.7e-09, sextile 1 p = 0.18, sextile 6 p = 1.8e-05). With one value per cycle, the population decrease holds (lower in 20 of 30 cycles, p = 0.016) and so does sextile 6 (p = 0.048), but sextiles 4 and 5 do not (p = 0.87 and 0.12), so the sextile results rest on pooling cells within cycles.

## fig3b-sextiles-over-sleep

> Over SLEEP, restricted to nonREM, the fastest sextile decreases its rate while the slowest shows
> a marginal increase.

In [ ]:
for rd in ("a", "b"):
    print("Reading (%s):" % rd)
    for g in range(1, 7):
        r, p, n = pooled_r(pts[rd], [g])
        print("  sextile %d: R = %+.3f, p = %.2g, points %d" % (g, r, p, n))
print("\nPer subject, reading (a), sextiles 1 and 6:")
for subj in sorted(tab.subject.unique()):
    sub = [t_ for t_ in pts["a"] if t_[2].subject == subj]
    if sub:
        out = []
        for g in (1, 6):
            try:
                out.append("%+.3f (p %.2g)" % pooled_r(sub, [g])[:2])
            except ValueError:
                out.append("n/a")
        print("  %s: sextile 1 %s, sextile 6 %s" % (subj, *out))

**Result: reproduced.** Per packet over SLEEP, sextile 6 decreases (R = -0.447, p = 3.1e-11) and sextile 1 increases (R = +0.163, p = 0.0074); sextile 5 also decreases and sextile 2 increases. In bins, sextile 6 gives R = -0.430 and sextile 1 R = +0.094 (p = 0.11). Sextile 6 has a negative R in all 7 subjects where it can be computed.

## fig3b-cv-over-sleep

> The coefficient of variation of pE firing rates decreases over the course of SLEEP (R = -0.05,
> p < 0.01).

In [ ]:
def cv_sleep(pts_r):
    xs, ys = [], []
    for x, rmat, s in pts_r:
        mu = rmat.mean(axis=0); sd = rmat.std(axis=0)
        ok = mu > 0
        xs.append(x[ok]); ys.append(sd[ok] / mu[ok])
    return tuple(st.pearsonr(np.concatenate(xs), np.concatenate(ys))) + (len(np.concatenate(xs)),)

for rd in ("a", "b"):
    print("Reading (%s): R = %.3f, p = %.2g, points %d" % (rd, *cv_sleep(pts[rd])))

**Result: not reproduced.** The CV of pE rates does not change detectably over SLEEP: R = -0.026 (p = 0.66) per packet and R = -0.014 (p = 0.81) in bins.

## Time courses within nonREM packets and episodes

The next claims use the same time courses: 10 bins of normalized time within packets of at least
20 s and within nonREM episodes of at least 20 s (Non-REM time only).

In [ ]:
def intervals_of(kind, min_dur=20.0, sess=None):
    """(session, span start, span stop, included intervals) for packets, episodes, WAKE or REM epochs,
    over the sessions in `sess` (all by default)."""
    out = []
    for s in (sessions if sess is None else sess):
        if kind in ("packet", "WAKE", "REM"):
            lab = "NREM" if kind == "packet" else kind
            for i in np.where(s.state == lab)[0]:
                if s.stops[i] - s.starts[i] >= min_dur:
                    out.append((s, s.starts[i], s.stops[i], None, [i]))
        else:
            for a, b in ss.nrem_episodes(s.state):
                if s.stops[b] - s.starts[a] >= min_dur:
                    rows_ = [i for i in range(a, b + 1) if s.state[i] == "NREM"]
                    out.append((s, s.starts[a], s.stops[b], [(s.starts[i], s.stops[i]) for i in rows_], rows_))
    return out

def course(kind, nbins=10, min_dur=20.0, sess=None):
    """Per interval: bin centres, pE rate matrix, burst counts per bin (short, all), session."""
    res = []
    for s, a, b, inc, rows_ in intervals_of(kind, min_dur, sess):
        bins = ss.binned_intervals(a, b, nbins, inc)
        rmat, dur = ss.bin_rates(s, s.pe, bins)
        ok = dur > 0
        # burst: intervals within the same packet (or epoch), later spike binned
        pk_s = np.array([s.starts[i] for i in rows_]); pk_e = np.array([s.stops[i] for i in rows_])
        short = np.zeros(nbins); alln = np.zeros(nbins)
        for u in s.pe:
            isi, t2, _ = ss.isis_in(s.spikes[u], pk_s, pk_e)
            k = np.clip(((t2 - a) / (b - a) * nbins).astype(int), 0, nbins - 1)
            alln += np.bincount(k, minlength=nbins)
            short += np.bincount(k[isi < ss.BURST_ISI], minlength=nbins)
        x = (np.arange(nbins) + 0.5) / nbins
        res.append((x[ok], rmat[:, ok], s, short[ok], alln[ok]))
    return res

_cache = {}
def course_c(kind, nbins=10, min_dur=20.0):
    if (kind, nbins, min_dur) not in _cache:
        _cache[(kind, nbins, min_dur)] = course(kind, nbins, min_dur)
    return _cache[(kind, nbins, min_dur)]

courses = {k: course_c(k) for k in ("episode", "packet")}
print({k: len(v) for k, v in courses.items()}, "intervals")
print("Packets of at least 20 s by subject:", collections.Counter(c[2].subject for c in courses["packet"]))

## fig4iv-sextiles-within-nrem

> Within both nonREM episodes and nonREM packets, the fastest sextiles decrease their rates while
> the slowest increase.

In [ ]:
def sextile_table(cs):
    out = {}
    for g in range(1, 7):
        out[g] = pooled_r([(x, r, s) for x, r, s, *_ in cs], [g])
    out["all pE"] = pooled_r([(x, r, s) for x, r, s, *_ in cs], None)
    return out

for kind in ("episode", "packet"):
    t = sextile_table(courses[kind])
    print("%s: " % kind + "; ".join("%s R = %+.3f (p %.2g)" % (("sextile %d" % g) if g != "all pE" else g, v[0], v[1])
                                     for g, v in t.items()))
print("\nVariants (sextile 1 and sextile 6 R, p):")
for kind in ("episode", "packet"):
    for nb, md in ((5, 20.0), (20, 20.0), (10, 60.0)):
        t = sextile_table(course_c(kind, nb, md))
        print("  %s, %d bins, min %d s: s1 %+.3f (p %.2g), s6 %+.3f (p %.2g)" % (kind, nb, md, *t[1][:2], *t[6][:2]))
print("\nPer subject, packets, sextile 1 and 6:")
for subj in sorted(tab.subject.unique()):
    cs = [c for c in courses["packet"] if c[2].subject == subj]
    out = []
    for g in (1, 6):
        try:
            out.append("%+.3f (p %.2g, n %d)" % pooled_r([(x, r, s) for x, r, s, *_ in cs], [g]))
        except ValueError:
            out.append("n/a")
    print("  %s: s1 %s, s6 %s" % (subj, *out))
fig, ax = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
for j, kind in enumerate(("episode", "packet")):
    for g, c in zip(range(1, 7), plt.cm.viridis(np.linspace(0, 1, 6))):
        vals = []
        for x, r, s, *_ in courses[kind]:
            if len(x) == 10:
                v = group_mean(r, sext[s.path][1] == g)
                if v is not None:
                    vals.append(v)
        m = np.nanmean(vals, axis=0)
        ax[j].plot(np.arange(10) + 0.5, m, color=c, label=f"sextile {g}")
        print("%s, sextile %d: mean normalized rate per bin from %.3f to %.3f" % (kind, g, np.nanmin(m), np.nanmax(m)))
    ax[j].set_title(kind); ax[j].set_xlabel("normalized time (bin)")
ax[0].set_ylabel("rate / interval mean"); ax[1].legend(fontsize=7)
plt.show()
print("\nPer session, R of sextiles 1 and 6 over normalized time (pooled within the session); Wilcoxon over sessions:")
for kind in ("episode", "packet"):
    for g in (1, 6):
        rs = []
        for sid in sorted({c[2].session_id for c in courses[kind]}):
            try:
                rs.append(pooled_r([(x, r, s) for x, r, s, *_ in courses[kind] if s.session_id == sid], [g])[0])
            except ValueError:
                pass
        rs = np.array(rs)
        rs = rs[np.isfinite(rs)]
        print("  %s, sextile %d: %d sessions, median R %+.3f, positive in %d, Wilcoxon p = %.2g" % (
            kind, g, len(rs), np.median(rs), (rs > 0).sum(), st.wilcoxon(rs).pvalue))

**Result: reproduced.** Within episodes sextile 6 decreases (R = -0.083, p = 3.7e-08) and sextile 1 increases (R = +0.060, p = 6.3e-06); within packets R = -0.055 (p = 4.4e-06) and +0.061 (p = 2e-08). The signs hold for 5 and 20 bins and for a 60 s minimum. The effects are small (the mean normalized rate of each sextile stays within 0.92 to 1.15 of the interval mean) and differ between subjects: within packets sextile 6 increases in Rizzo (R = +0.121) and BWRat19 and BWRat20 show no decrease. With one R per session, sextile 1 increases in episodes (p = 0.049) and packets (p = 0.011) and sextile 6 decreases in episodes (p = 0.013), but not in packets (median R -0.025, 6 of 16 sessions positive, p = 0.19).

## fig4v-cv-within-nrem

> The coefficient of variation of binned population rates decreases within nonREM episodes
> (R = -0.13) and packets (R = -0.12), both p < 1e-10.

In [ ]:
def cv_r(cs):
    xs, ys = [], []
    for x, r, s, *_ in cs:
        mu, sd = r.mean(axis=0), r.std(axis=0)
        ok = mu > 0
        xs.append(x[ok]); ys.append(sd[ok] / mu[ok])
    return st.pearsonr(np.concatenate(xs), np.concatenate(ys)) + (len(np.concatenate(xs)),)

for kind in ("episode", "packet"):
    r, p, n = cv_r(courses[kind])
    print("%s: R = %.3f, p = %.2g, points %d" % (kind, r, p, n))
    for nb, md in ((5, 20.0), (20, 20.0), (10, 60.0)):
        print("   variant %d bins, min %d s: R = %.3f, p = %.2g, points %d" % (nb, md, *cv_r(course_c(kind, nb, md))))

**Result: not reproduced.** The CV of binned pE rates does not decrease within episodes (R = -0.005, p = 0.71) or packets (R = 0.011, p = 0.28), nor in any variant (largest absolute R 0.011).

## fig4vi-burst-within-nrem

> The 15 ms burst index of pE cells increases within nonREM episodes (R = 0.09) and packets
> (R = 0.08), both p < 1e-10.

In [ ]:
def burst_r(cs):
    xs, ys = [], []
    for x, r, s, short, alln in cs:
        ok = alln > 0
        xs.append(x[ok]); ys.append(short[ok] / alln[ok])
    return st.pearsonr(np.concatenate(xs), np.concatenate(ys)) + (len(np.concatenate(xs)),)

for kind in ("episode", "packet"):
    r, p, n = burst_r(courses[kind])
    print("%s: R = %.3f, p = %.2g, points %d" % (kind, r, p, n))
    for nb, md in ((5, 20.0), (20, 20.0), (10, 60.0)):
        print("   variant %d bins, min %d s: R = %.3f, p = %.2g, points %d" % (nb, md, *burst_r(course_c(kind, nb, md))))

**Result: not reproduced.** The burst index rises slightly within episodes (R = 0.029, p = 0.022) and packets (R = 0.024, p = 0.021), far from the paper's p < 1e-10; the variants give R = 0.020 to 0.047. The direction is as reported, the size is not.

## fig4i-band-power-within-nrem

> Delta and sigma power increase over nonREM episodes and packets, while gamma power decreases over
> episodes but not over packets.

In [ ]:
def lfp_course(kind, nbins=10, min_dur=20.0, sess=None):
    out = {b: ([], []) for b in ss.BANDS}
    for s, a, b, inc, rows_ in intervals_of(kind, min_dur, sess):
        rows_ = [i for i in rows_ if i in s.lfp]
        if not rows_:
            continue
        t = np.concatenate([s.lfp[i][0] for i in rows_])
        k = np.clip(((t - a) / (b - a) * nbins).astype(int), 0, nbins - 1)
        for band in ss.BANDS:
            v = np.concatenate([s.lfp[i][1][band] for i in rows_])
            m = np.array([v[k == j].mean() if np.any(k == j) else np.nan for j in range(nbins)])
            ok = np.isfinite(m)
            if ok.sum() < 2:
                continue
            out[band][0].append(((np.arange(nbins) + 0.5) / nbins)[ok]); out[band][1].append(m[ok] - m[ok].mean())
    return {band: st.pearsonr(np.concatenate(x), np.concatenate(y)) + (len(np.concatenate(x)),) for band, (x, y) in out.items()}

for kind in ("episode", "packet"):
    res = lfp_course(kind)
    print("%s: " % kind + "; ".join("%s R = %+.3f (p %.2g, n %d)" % (b, *v) for b, v in res.items()))
    for nb, md in ((5, 20.0), (20, 20.0), (10, 60.0)):
        res = lfp_course(kind, nb, md)
        print("   variant %d bins, min %d s: " % (nb, md) + "; ".join("%s %+.3f (p %.2g)" % (b, v[0], v[1]) for b, v in res.items()))
print("\nPer subject, packets:")
for subj in sorted(tab.subject.unique()):
    res = lfp_course("packet", sess=[s for s in sessions if s.subject == subj])
    print("  %s: " % subj + "; ".join("%s %+.3f (p %.2g)" % (b, v[0], v[1]) for b, v in res.items()))

**Result: not reproduced.** Sigma power increases within episodes and packets (R = 0.135 and 0.104, both p < 1e-20), and delta power increases within episodes (R = 0.069, p = 1.1e-07) but not within packets (R = -0.005, p = 0.61). Gamma power increases within episodes (R = +0.062) and packets (R = +0.182), the opposite of the reported decrease over episodes. One channel per session (the conversion's spindle reference) is used, which may differ from the paper's.

## fig4-delta-packet-length

> Mean delta power within whole packets, normalized by nonREM delta power, is positively correlated
> with packet length (R = 0.31, p < 0.05).

In [ ]:
pk = []
for s in sessions:
    if not s.lfp:
        continue
    allv = {b: np.concatenate([10 ** s.lfp[i][1][b] for i in s.lfp]) for b in ss.BANDS}
    for i, (t, pw) in s.lfp.items():
        if s.stops[i] - s.starts[i] < 20:
            continue
        rr = {}
        for g in (1, 6):
            units = s.pe[sext[s.path][1] == g]
            rr[g] = np.mean(ss.rates(s.spikes, units, s.starts[i], s.stops[i])) if len(units) else np.nan
        last = t >= s.stops[i] - 20
        pk.append(dict(subject=s.subject, session=s.session_id, row=i, dur=s.stops[i] - s.starts[i],
                       delta_norm=np.mean(10 ** pw["delta"]) / allv["delta"].mean(),
                       delta=pw["delta"].mean(), gamma=pw["gamma"].mean(), sigma_last20=pw["sigma"][last].mean(),
                       s1=rr[1], s6=rr[6]))
pk = pd.DataFrame(pk)
r, p = st.pearsonr(pk.dur, pk.delta_norm)
print("Packets of at least 20 s with LFP: %d; R = %.3f, p = %.2g; within 0.279-0.341: %s" % (len(pk), r, p, 0.279 <= r <= 0.341))
print("Spearman rho = %.3f; R with log10 duration = %.3f" % (st.spearmanr(pk.dur, pk.delta_norm)[0], st.pearsonr(np.log10(pk.dur), pk.delta_norm)[0]))
print("Normalized delta power: %.2f to %.2f; durations %.0f to %.0f s" % (pk.delta_norm.min(), pk.delta_norm.max(), pk.dur.min(), pk.dur.max()))
print("Per subject R:", {subj: round(st.pearsonr(g.dur, g.delta_norm)[0], 3) for subj, g in pk.groupby("subject")})

**Result: reproduced.** Over 915 packets, normalized delta power correlates with packet length, R = 0.334 (p = 2.8e-25), within 0.279 to 0.341, near its upper edge. The correlation is positive in 9 of 10 subjects.

## fig4cd-packet-rate-lfp

> Per packet, the rate of the fastest sextile is negatively correlated with delta power (R = -0.28)
> and positively with gamma power (R = 0.44), and the rate of the slowest sextile is positively
> correlated with sigma power in the last 20 s of the packet (R = 0.25), all p < 1e-10.

In [ ]:
z = lambda v: (v - v.mean()) / v.std() if v.std() > 0 else v * np.nan
pz = pk.copy()
for c in ("delta", "gamma", "sigma_last20", "s1", "s6"):
    pz[c] = pk.groupby("session")[c].transform(z)
out = {}
for name, a, b, exp_, tol in (("s6 vs delta", "s6", "delta", -0.28, 0.028), ("s6 vs gamma", "s6", "gamma", 0.44, 0.044),
                              ("s1 vs sigma last 20 s", "s1", "sigma_last20", 0.25, 0.025)):
    d = pz[[a, b]].dropna()
    r, p = st.pearsonr(d[a], d[b])
    out[name] = (r, p)
    print("%s: R = %.3f, p = %.2g, n %d; within tolerance %s; sign as stated %s" % (
        name, r, p, len(d), abs(r - exp_) <= tol, np.sign(r) == np.sign(exp_)))
print("\nPer subject:")
for subj, g in pz.groupby("subject"):
    vals = []
    for a, b in (("s6", "delta"), ("s6", "gamma"), ("s1", "sigma_last20")):
        d = g[[a, b]].dropna()
        vals.append(st.pearsonr(d[a], d[b])[0] if len(d) > 2 else np.nan)
    print("  %s: s6-delta %.3f, s6-gamma %.3f, s1-sigma %.3f, packets %d" % (subj, *vals, len(g)))

**Result: partially reproduced.** All three correlations have the stated sign: sextile 6 rate against delta power R = -0.416 (p = 1.3e-28) and against gamma power R = 0.570 (p = 4.2e-57), stronger than the paper's -0.28 and 0.44, and sextile 1 rate against sigma power in the last 20 s R = 0.104 (p = 0.0029), weaker than 0.25. None is within tolerance.

## State triplets

Triplets of consecutive epochs A, B, A in the state table; the change per pE cell is its rate in the
second A minus the first.

In [ ]:
def triplets(a_state, b_state):
    out = []
    for s in sessions:
        for i in range(1, len(s.state) - 1):
            if s.state[i - 1] == a_state and s.state[i] == b_state and s.state[i + 1] == a_state:
                r1 = ss.rates(s.spikes, s.pe, s.starts[i - 1], s.stops[i - 1])
                r2 = ss.rates(s.spikes, s.pe, s.starts[i + 1], s.stops[i + 1])
                for k, u in enumerate(s.pe):
                    out.append(dict(subject=s.subject, session=s.session_id, row=i, unit=u, sextile=sext[s.path][1][k],
                                    pre=r1[k], post=r2[k], ch=r2[k] - r1[k], mid_dur=s.stops[i] - s.starts[i]))
    return pd.DataFrame(out)

def wp(v, **kw):
    """Wilcoxon signed-rank p, or NaN when there is nothing to test."""
    v = np.asarray(v, float)
    return st.wilcoxon(v, **kw).pvalue if len(v) and np.any(v != 0) else np.nan

def triplet_report(d, alpha):
    print("triplets %d, cell-triplet pairs %d" % (d.groupby(["session", "row"]).ngroups, len(d)))
    res = {}
    for g in range(1, 7):
        x = d[d.sextile == g]
        p = st.wilcoxon(x.ch).pvalue if (x.ch != 0).any() else 1.0
        cm = x.groupby(["session", "unit"]).ch.mean()
        pc = st.wilcoxon(cm).pvalue if (cm != 0).any() else 1.0
        res[g] = (x.ch.mean(), p)
        print("  sextile %d: n %d, mean change %+.4f Hz (pre mean %.3f), Wilcoxon p = %.2g%s | per-cell means: n %d, mean %+.4f, p = %.2g" % (
            g, len(x), x.ch.mean(), x.pre.mean(), p, " *" if p < alpha else "", len(cm), cm.mean(), pc))
    print("  one value per triplet (mean over its cells), and per session; Wilcoxon over these:")
    for g in range(1, 7):
        x = d[d.sextile == g]
        t_, s_ = x.groupby(["session", "row"]).ch.mean(), x.groupby("session").ch.mean()
        print("    sextile %d: triplets %d, mean %+.4f, p = %.2g | sessions %d, mean %+.4f, p = %.2g, lower in %d" % (
            g, len(t_), t_.mean(), wp(t_), len(s_), s_.mean(), wp(s_), (s_ < 0).sum()))
    return res

def triplet_by_subject(d, gs):
    print("  per subject, mean change (Hz) for sextiles %s:" % (gs,))
    print(d[d.sextile.isin(gs)].groupby(["subject", "sextile"]).ch.mean().unstack().round(4).to_string())

trip = {k: triplets(*k) for k in (("MA", "NREM"), ("NREM", "WAKE"), ("NREM", "MA"), ("NREM", "REM"))}

## fig5b-nrem-between-ma

> An intervening nonREM packet reduces the rates of the top two sextiles from one MA to the next
> (both p < 0.01).

In [ ]:
res_ma = triplet_report(trip[("MA", "NREM")], 0.01)
triplet_by_subject(trip[("MA", "NREM")], [5, 6])

**Result: not reproduced.** Across MA, packet, MA triplets (148 triplets), sextile 6 changes by -0.104 Hz (p = 0.077) and sextile 5 by -0.011 Hz (p = 0.24), neither at p < 0.01. With per-cell means sextile 6 decreases at p = 0.029. The direction for sextile 6 is as reported. With one value per triplet or per session neither sextile changes at p < 0.1.

## fig5b-wake-between-packets

> An intervening WAKE episode increases the rate of the fastest sextile from one nonREM packet to
> the next (24 triplets).

In [ ]:
res_w = triplet_report(trip[("NREM", "WAKE")], 0.05)
d = trip[("NREM", "WAKE")]
print("WAKE durations in these triplets (s): median %.0f, range %.0f to %.0f" % (
    d.groupby(["session", "row"]).mid_dur.first().median(), d.mid_dur.min(), d.mid_dur.max()))
triplet_by_subject(d, [1, 6])
print("\nTriplets and the change of sextile 6 by the minimum duration of the intervening WAKE:")
for mn in (40, 60, 120, 180, 300, 420, 600, 900):
    x = d[d.mid_dur >= mn]
    x6 = x[x.sextile == 6]
    print("  WAKE of at least %d s: triplets %d, sextile 6 n %d, mean change %+.4f Hz, Wilcoxon p = %.2g" % (
        mn, x.groupby(["session", "row"]).ngroups, len(x6), x6.ch.mean(), wp(x6.ch)))

**Result: not reproduced.** Across 265 packet, WAKE, packet triplets (the paper had 24), sextile 6 does not increase: its mean change is -0.051 Hz (p = 0.092). Sextiles 1 and 3 increase slightly (p = 0.0063 and 0.015). The WAKE definition here admits every Awake epoch of at least 40 s between packets (median 102 s), which likely differs from the paper's selection. No minimum WAKE duration from 40 to 900 s gives a sextile 6 increase at p < 0.05; WAKE of at least 900 s leaves 23 triplets, near the paper's 24, with a mean change of -0.185 Hz (p = 0.5).

## fig5b-ma-between-packets

> An intervening MA increases the rates of slow-firing pE cells from one packet to the next and does
> not affect fast-firing cells.

In [ ]:
res_m = triplet_report(trip[("NREM", "MA")], 0.05)
triplet_by_subject(trip[("NREM", "MA")], [1, 6])

**Result: not reproduced.** Across 391 packet, MA, packet triplets, sextile 1 does not increase significantly (+0.0034 Hz, p = 0.082), and sextiles 5 and 6 decrease (p = 0.0023 and 6.8e-05), where the paper reports no effect on fast cells. Sextile 2 increases (p = 0.00019). With one value per triplet sextile 1 increases at p = 0.037, but per session it does not (p = 0.17), while the decrease of sextiles 5 and 6 holds per session (p = 0.0094 and 0.00015).

## fig5b-rem-between-packets

> An intervening REM episode decreases the rates of all sextiles from one packet to the next.

In [ ]:
res_r = triplet_report(trip[("NREM", "REM")], 0.05)
triplet_by_subject(trip[("NREM", "REM")], [1, 6])
print("REM epochs followed by Non-REM: %d of %d REM epochs" % (
    sum(int(np.sum((s.state[:-1] == "REM") & (s.state[1:] == "NREM"))) for s in sessions),
    sum(int(np.sum(s.state == "REM")) for s in sessions)))

**Result: not reproduced.** Only 17 packet, REM, packet triplets exist, as only 49 of 534 REM epochs are followed directly by Non-REM. Sextiles 4 and 5 decrease (p = 0.022 and 0.011) and the others do not change significantly; sextiles 1 and 2 have positive mean changes. Two of six sextiles is below the three needed for a partial result. With one value per triplet only sextile 4 decreases at p < 0.05.

## fig5c-rem-duration

> Longer REM episodes produce a proportionally larger decrease of rates from one packet to the next
> (R = -0.22, p < 1e-2).

In [ ]:
d = trip[("NREM", "REM")]
per = d.groupby(["session", "row"]).agg(pre=("pre", "mean"), post=("post", "mean"), dur=("mid_dur", "first"),
                                        subject=("subject", "first"))
per = per[per.pre > 0]
ra = st.pearsonr(per.dur, (per.post - per.pre) / per.pre)
dc = d[d.pre > 0]
rb = st.pearsonr(dc.mid_dur, (dc.post - dc.pre) / dc.pre)
print("(a) per triplet: R = %.3f, p = %.2g, n %d | (b) per cell and triplet: R = %.3f, p = %.2g, n %d" % (
    ra[0], ra[1], len(per), rb[0], rb[1], len(dc)))
print("Within -0.242 to -0.198:", {"a": -0.242 <= ra[0] <= -0.198, "b": -0.242 <= rb[0] <= -0.198})
print("REM durations %.0f to %.0f s; fractional change (a) %.2f to %.2f" % (
    per.dur.min(), per.dur.max(), ((per.post - per.pre) / per.pre).min(), ((per.post - per.pre) / per.pre).max()))
print("Spearman (a): rho = %.3f, p = %.2g" % st.spearmanr(per.dur, (per.post - per.pre) / per.pre))

**Result: not reproduced.** The fractional rate change across REM is not correlated with REM duration: R = -0.047 (p = 0.86) over 17 triplets, and R = -0.061 (p = 0.21) over 436 cell and triplet pairs. Without BWRat21-121813, whose state labels could not be checked against the spikes, 13 triplets give R = -0.452 (p = 0.12).

## fig5c-sigma-ma

> Sigma power in the last 20 s of a packet between two MAs correlates with a larger increase of the
> slowest sextile from one MA to the next (R = 0.181, p < 0.01).

In [ ]:
sig_all = {}
for s in sessions:
    if not s.lfp:
        continue
    v = {i: s.lfp[i][1]["sigma"][s.lfp[i][0] >= s.stops[i] - 20].mean() for i in s.lfp}
    vals = np.array(list(v.values()))
    sig_all[s.path] = {i: (x - np.nanmean(vals)) / np.nanstd(vals) for i, x in v.items()}
d = trip[("MA", "NREM")]
d1 = d[d.sextile == 1].groupby(["session", "row"]).agg(ch=("ch", "mean")).reset_index()
path_of = {s.session_id: s.path for s in sessions}
d1["sigma"] = [sig_all.get(path_of[a], {}).get(b, np.nan) for a, b in zip(d1.session, d1.row)]
d1 = d1.dropna()
r, p = st.pearsonr(d1.sigma, d1.ch)
print("MA-packet-MA triplets with sextile 1 cells and LFP: %d; R = %.3f, p = %.2g; within 0.163-0.199: %s" % (len(d1), r, p, 0.163 <= r <= 0.199))
print("Spearman rho = %.3f, p = %.2g" % st.spearmanr(d1.sigma, d1.ch))

**Result: not reproduced.** Over 125 MA, packet, MA triplets, sigma power in the last 20 s of the packet is not correlated with the change of sextile 1 from one MA to the next (R = 0.033, p = 0.71).

## fig5c-delta-change

> The increase of the slowest sextile's rate from one packet to the next is positively correlated
> with the change of delta power between the packets, regardless of the intervening state.

In [ ]:
dd = []
for sess, g in pk.groupby("session", sort=False):
    g = g.sort_values("row")
    dd.append(pd.DataFrame({"subject": g.subject.values[1:], "session": sess, "d_s1": np.diff(g.s1.values), "d_delta": np.diff(g.delta_norm.values),
                            "d_s6": np.diff(g.s6.values)}))
dd = pd.concat(dd).dropna()
r, p = st.pearsonr(dd.d_delta, dd.d_s1)
print("Consecutive packet pairs (both at least 20 s): %d; sextile 1: R = %.3f, p = %.2g" % (len(dd), r, p))
print("Sextile 6 for comparison: R = %.3f, p = %.2g" % st.pearsonr(dd.d_delta, dd.d_s6))
print("Per subject, sextile 1 R:", {subj: round(st.pearsonr(g.d_delta, g.d_s1)[0], 3) for subj, g in dd.groupby("subject") if len(g) > 2})

**Result: reproduced.** Over 613 pairs of consecutive packets, the change of sextile 1 rate correlates positively with the change of normalized delta power (R = 0.183, p = 5.3e-06), while sextile 6 correlates negatively (R = -0.368). Per subject the sextile 1 correlation is positive in 4 of 8 subjects, so the pooled result rests on some animals.

## figs5-within-wake-rem

> Within WAKE the two fastest sextiles increase and the two slowest decrease their rates; within REM
> all sextiles except the fourth increase.

In [ ]:
wk = course("WAKE", 10, 40.0)
rm = course("REM", 10, 20.0)
tw, trm = sextile_table(wk), sextile_table(rm)
print("WAKE epochs %d, REM epochs %d" % (len(wk), len(rm)))
for name, t in (("WAKE", tw), ("REM", trm)):
    print(name + ": " + "; ".join("%s R = %+.3f (p %.2g)" % (g, v[0], v[1]) for g, v in t.items()))
wake_ok = all(tw[g][0] > 0 and tw[g][1] < 0.05 for g in (5, 6)) and all(tw[g][0] < 0 and tw[g][1] < 0.05 for g in (1, 2))
rem_ok = all(trm[g][0] > 0 and trm[g][1] < 0.05 for g in (1, 2, 3, 5, 6))
print("WAKE part holds:", wake_ok, "| REM part holds:", rem_ok)

**Result: not reproduced.** Within 495 WAKE epochs, sextiles 5 and 6 decrease (R = -0.122 and -0.120, both p < 1e-12), the opposite of the reported increase, and sextiles 1 and 2 do not decrease (sextile 2 increases, R = +0.067). Within 388 REM epochs no sextile changes at p < 0.05 (largest R +0.028).

## Sessions whose clocks could not be confirmed

The data check could not confirm the state labels against the spikes in one session, and the LFP
against the spikes in seven. The cell below recomputes the main numbers of every tested claim with
all 21 sessions, without the session whose labels were not confirmed, and without the seven sessions
whose LFP was not confirmed. The sextile edges stay those of all 21 sessions. The first column repeats
the numbers above.

In [ ]:
drop_labels = sorted(str(x) for x in clock.session[~clock["labels consistent"]])
drop_lfp = sorted(str(x) for x in clock.session[~clock["lfp consistent"]])
all_ids = set(tab.session)
subsets = {"all 21": all_ids, "without " + ", ".join(drop_labels): all_ids - set(drop_labels),
           "without %d (LFP)" % len(drop_lfp): all_ids - set(drop_lfp)}
print("Labels not confirmed:", drop_labels, "| LFP not confirmed:", drop_lfp)
acg_lags = np.arange(300) + 0.5

def headlines(ids):
    S = [s for s in sessions if s.session_id in ids]
    out = []

    def add(claim, name, fn):
        try:
            v = float(fn())
        except (ValueError, KeyError, IndexError, TypeError):
            v = np.nan
        out.append((claim, name, v))

    c = cells_a[cells_a.session.isin(ids)]
    pe = c[c.cls == "pE"]
    add("dataset-composition", "pE cells", lambda: len(pe))
    add("dataset-composition", "pI cells", lambda: (c.cls == "pI").sum())
    for cls in ("pE", "pI"):
        for state in ("WAKE", "NREM", "REM"):
            add("fig2a-median-rates", "%s %s median (Hz)" % (cls, state), lambda: c[c.cls == cls][state].median())
    add("fig2a-skewed-wide-distribution", "WAKE skewness", lambda: st.skew(pe.WAKE))
    add("fig2a-skewed-wide-distribution", "WAKE 2.5th percentile (Hz)", lambda: np.percentile(pe.WAKE, 2.5))
    add("fig2a-skewed-wide-distribution", "WAKE 97.5th percentile (Hz)", lambda: np.percentile(pe.WAKE, 97.5))
    for state in ("WAKE", "REM", "NREM"):
        add("fig2a-fraction-above-2hz", "%% above 2 Hz, %s" % state, lambda: 100 * np.mean(pe[state] > 2))
    for other in ("WAKE", "REM"):
        a_, b_, n_ = (pe.NREM > 2).sum(), (pe[other] > 2).sum(), len(pe)
        add("fig2a-fraction-above-2hz", "chi-square p, NREM against %s" % other,
            lambda: st.chi2_contingency([[a_, n_ - a_], [b_, n_ - b_]])[1])
    k = (pe.WAKE > 0) & (pe.NREM > 0)
    f2b = ols(np.log10(pe.WAKE[k]), np.log10(pe.NREM[k]))
    add("fig2b-wake-nrem-slope", "R, log10", lambda: f2b[3])
    add("fig2b-wake-nrem-slope", "slope, log10", lambda: f2b[0])
    add("fig2b-wake-nrem-slope", "slope, upper 95% bound", lambda: f2b[2])
    bb = bi[bi.session.isin(ids)]
    for rd in ("a", "b"):
        for state in ("NREM", "WAKE", "REM"):
            add("fig2-burst-index-by-state", "median %% %s (%s)" % (state, rd), lambda: 100 * bb["%s_%s" % (state, rd)].median())
    add("fig2-burst-index-by-state", "Kruskal-Wallis p (a)",
        lambda: st.kruskal(*[bb[x + "_a"].dropna() for x in ("NREM", "WAKE", "REM")]).pvalue)
    for state in ("NREM", "REM", "WAKE"):
        mean = np.mean([h for h, sid in zip(acg[state], acg_sess[state]) if sid in ids], axis=0)
        sm = np.convolve(mean, np.ones(21) / 21, mode="same")
        w = (acg_lags >= 60) & (acg_lags <= 250)
        if state == "NREM":
            add("fig2c-acg-peaks", "NREM peak 2-50 ms (ms)", lambda: acg_lags[np.argmax(mean[2:50]) + 2])
        else:
            add("fig2c-acg-peaks", "%s smoothed maximum 60-250 ms (ms)" % state, lambda: acg_lags[w][np.argmax(sm[w])])
    g = pairs[pairs.session.isin(ids)].groupby("a").r.mean()
    add("fig2f-rate-group-correlations", "(6,6) mean r", lambda: g[(6, 6)])
    add("fig2f-rate-group-correlations", "(6,6) the largest (1 if so)", lambda: g.idxmax() == (6, 6))
    add("fig2f-rate-group-correlations", "(1,6) mean r", lambda: g[(1, 6)])
    m = ma[ma.session.isin(ids)]
    add("ma-statistics", "median duration (s)", lambda: np.median(np.concatenate([np.array(x, float) for x in m.durs])))
    add("ma-statistics", "per min of SLEEP", lambda: m.freq.mean())
    add("ma-statistics", "% of SLEEP", lambda: m.frac.mean())
    F = fr[np.isin(fr_sess, list(ids))]
    add("ma-fraction-over-nrem-episodes", "R of bin means", lambda: st.pearsonr(np.arange(10) + 0.5, F.mean(axis=0))[0])
    add("ma-fraction-over-nrem-episodes", "R pooled", lambda: st.pearsonr(np.tile(np.arange(10) + 0.5, len(F)), F.ravel())[0])
    add("packet-duration", "median (s)", lambda: np.median(np.concatenate([(s.stops - s.starts)[s.state == "NREM"] for s in S])))
    add("wake-sleep-cycles", "cycles per session", lambda: nc[nc.index.isin(ids)].mean())
    P = {rd: [t_ for t_ in pts[rd] if t_[2].session_id in ids] for rd in ("a", "b")}
    for rd in ("a", "b"):
        add("fig3b-population-decline", "R (%s)" % rd, lambda: pooled_r(P[rd])[0])
    f_ = fl[fl.session.isin(ids)]
    add("fig3-first-last-packet", "first mean (Hz)", lambda: f_["first"].mean())
    add("fig3-first-last-packet", "last mean (Hz)", lambda: f_["last"].mean())
    add("fig3-first-last-packet", "Wilcoxon p", lambda: st.wilcoxon(f_["first"], f_["last"]).pvalue)
    k = (f_["first"] > 0) & (f_["last"] > 0)
    f3d = ols(np.log10(f_["first"][k]), np.log10(f_["last"][k]))
    add("fig3d-first-last-slope", "slope, log10", lambda: f3d[0])
    add("fig3d-first-last-slope", "slope, upper 95% bound", lambda: f3d[2])
    b_ = ba[ba.session.isin(ids)]
    add("fig3e-wake-before-after", "Wilcoxon p, all pE", lambda: st.wilcoxon(b_.pre, b_.post).pvalue)
    for gg in (1, 4, 5, 6):
        x = b_[b_.sextile == gg]
        add("fig3e-wake-before-after", "sextile %d one-tailed p" % gg,
            lambda: st.wilcoxon(x.post, x.pre, alternative="greater" if gg == 1 else "less").pvalue)
    for gg in (1, 6):
        add("fig3b-sextiles-over-sleep", "sextile %d R (a)" % gg, lambda: pooled_r(P["a"], [gg])[0])
        add("fig3b-sextiles-over-sleep", "sextile %d p (a)" % gg, lambda: pooled_r(P["a"], [gg])[1])
    add("fig3b-cv-over-sleep", "R (a)", lambda: cv_sleep(P["a"])[0])
    add("fig3b-cv-over-sleep", "p (a)", lambda: cv_sleep(P["a"])[1])
    for kind in ("episode", "packet"):
        cs = [c_ for c_ in courses[kind] if c_[2].session_id in ids]
        try:
            t6 = sextile_table(cs)
        except ValueError:
            t6 = {}
        for gg in (1, 6):
            add("fig4iv-sextiles-within-nrem", "%s, sextile %d R" % (kind, gg), lambda: t6[gg][0])
            add("fig4iv-sextiles-within-nrem", "%s, sextile %d p" % (kind, gg), lambda: t6[gg][1])
        add("fig4v-cv-within-nrem", "%s R" % kind, lambda: cv_r(cs)[0])
        add("fig4v-cv-within-nrem", "%s p" % kind, lambda: cv_r(cs)[1])
        add("fig4vi-burst-within-nrem", "%s R" % kind, lambda: burst_r(cs)[0])
        add("fig4vi-burst-within-nrem", "%s p" % kind, lambda: burst_r(cs)[1])
        lc = lfp_course(kind, sess=S)
        for band in ss.BANDS:
            add("fig4i-band-power-within-nrem", "%s, %s R" % (kind, band), lambda: lc[band][0])
            add("fig4i-band-power-within-nrem", "%s, %s p" % (kind, band), lambda: lc[band][1])
    q = pk[pk.session.isin(ids)]
    add("fig4-delta-packet-length", "R", lambda: st.pearsonr(q.dur, q.delta_norm)[0])
    add("fig4-delta-packet-length", "p", lambda: st.pearsonr(q.dur, q.delta_norm)[1])
    qz = pz[pz.session.isin(ids)]
    for a_, b2, nm in (("s6", "delta", "sextile 6 against delta R"), ("s6", "gamma", "sextile 6 against gamma R"),
                       ("s1", "sigma_last20", "sextile 1 against last-20-s sigma R")):
        add("fig4cd-packet-rate-lfp", nm, lambda: st.pearsonr(*qz[[a_, b2]].dropna().values.T)[0])
    for key, claim, gs in ((("MA", "NREM"), "fig5b-nrem-between-ma", (5, 6)), (("NREM", "WAKE"), "fig5b-wake-between-packets", (6,)),
                           (("NREM", "MA"), "fig5b-ma-between-packets", (1, 6)), (("NREM", "REM"), "fig5b-rem-between-packets", ())):
        dt = trip[key][trip[key].session.isin(ids)]
        add(claim, "triplets", lambda: dt.groupby(["session", "row"]).ngroups)
        for gg in gs:
            x = dt[dt.sextile == gg].ch
            add(claim, "sextile %d mean change (Hz)" % gg, lambda: x.mean())
            add(claim, "sextile %d p" % gg, lambda: wp(x))
        if key == ("NREM", "REM"):
            add(claim, "sextiles lower at p < 0.05", lambda: sum(
                dt[dt.sextile == g_].ch.mean() < 0 and wp(dt[dt.sextile == g_].ch) < 0.05 for g_ in range(1, 7)))
    dr = trip[("NREM", "REM")][trip[("NREM", "REM")].session.isin(ids)]
    per_ = dr.groupby(["session", "row"]).agg(pre=("pre", "mean"), post=("post", "mean"), dur=("mid_dur", "first"))
    per_ = per_[per_.pre > 0]
    add("fig5c-rem-duration", "R per triplet", lambda: st.pearsonr(per_.dur, (per_.post - per_.pre) / per_.pre)[0])
    add("fig5c-rem-duration", "p per triplet", lambda: st.pearsonr(per_.dur, (per_.post - per_.pre) / per_.pre)[1])
    e1 = d1[d1.session.isin(ids)]
    add("fig5c-sigma-ma", "R", lambda: st.pearsonr(e1.sigma, e1.ch)[0])
    add("fig5c-sigma-ma", "p", lambda: st.pearsonr(e1.sigma, e1.ch)[1])
    e2 = dd[dd.session.isin(ids)]
    add("fig5c-delta-change", "R", lambda: st.pearsonr(e2.d_delta, e2.d_s1)[0])
    add("fig5c-delta-change", "p", lambda: st.pearsonr(e2.d_delta, e2.d_s1)[1])
    for name_, cs_all in (("WAKE", wk), ("REM", rm)):
        try:
            tt = sextile_table([c_ for c_ in cs_all if c_[2].session_id in ids])
        except ValueError:
            tt = {}
        if name_ == "WAKE":
            for gg in (1, 2, 5, 6):
                add("figs5-within-wake-rem", "WAKE sextile %d R" % gg, lambda: tt[gg][0])
                add("figs5-within-wake-rem", "WAKE sextile %d p" % gg, lambda: tt[gg][1])
        else:
            add("figs5-within-wake-rem", "REM sextiles higher at p < 0.05",
                lambda: sum(tt[g_][0] > 0 and tt[g_][1] < 0.05 for g_ in range(1, 7)))
    return out

tables = {name: headlines(ids) for name, ids in subsets.items()}
first = next(iter(tables.values()))
sens = pd.DataFrame({name: [v for _, _, v in rows_] for name, rows_ in tables.items()},
                    index=pd.MultiIndex.from_tuples([(c_, q_) for c_, q_, _ in first], names=["claim", "quantity"]))
pos = {c_: i for i, c_ in enumerate(dict.fromkeys(c_ for c_, _, _ in first))}
sens = sens.iloc[sorted(range(len(first)), key=lambda i: pos[first[i][0]])]
print(sens.to_string(float_format=lambda v: "%.4g" % v))

Leaving out BWRat21-121813, whose state labels could not be checked against the spikes, changes no
judgment: the largest change is fig5c-rem-duration, where the 13 remaining triplets give R = -0.452
(p = 0.12). Leaving out the seven sessions whose LFP could not be checked changes no judgment of the
LFP claims (fig4i, fig4-delta-packet-length, fig4cd, fig5c-sigma-ma, fig5c-delta-change).
fig4-delta-packet-length moves from R = 0.334 to 0.297 and fig5c-delta-change from R = 0.183
(p = 5.3e-06) to 0.114 (p = 0.011); in fig4i, delta over packets turns from -0.005 to +0.014 (p = 0.27)
and gamma over episodes from +0.062 to +0.005 (p = 0.77), and both parts fail either way. The LFP clock
does not bear on the spike-only claims, but without these 222 pE cells two of them would be judged
differently: the last-packet mean of fig3-first-last-packet rises to 1.006 Hz, outside its tolerance,
and the nonREM autocorrelogram peak of fig2c-acg-peaks moves to 6.5 ms, outside 7 to 9 ms.

## Summary

| Claim | Outcome |
|---|---|
| dataset-composition | not reproduced |
| fig2a-median-rates | partially reproduced |
| fig2a-skewed-wide-distribution | partially reproduced |
| fig2a-fraction-above-2hz | not reproduced |
| fig2b-wake-nrem-slope | reproduced |
| fig2-burst-index-by-state | partially reproduced |
| fig2c-acg-peaks | partially reproduced |
| fig2c-acg-decay | blocked (methods underspecified) |
| fig2f-rate-group-correlations | partially reproduced |
| ma-statistics | not reproduced |
| ma-fraction-over-nrem-episodes | partially reproduced |
| packet-duration | reproduced |
| wake-sleep-cycles | not reproduced |
| fig3b-population-decline | partially reproduced |
| fig3-first-last-packet | reproduced |
| fig3d-first-last-slope | reproduced |
| fig3e-wake-before-after | partially reproduced |
| fig3b-sextiles-over-sleep | reproduced |
| fig3b-cv-over-sleep | not reproduced |
| fig4iv-sextiles-within-nrem | reproduced |
| fig4v-cv-within-nrem | not reproduced |
| fig4vi-burst-within-nrem | not reproduced |
| fig4i-band-power-within-nrem | not reproduced |
| fig4-delta-packet-length | reproduced |
| fig4cd-packet-rate-lfp | partially reproduced |
| fig5b-nrem-between-ma | not reproduced |
| fig5b-wake-between-packets | not reproduced |
| fig5b-ma-between-packets | not reproduced |
| fig5b-rem-between-packets | not reproduced |
| fig5c-rem-duration | not reproduced |
| fig5c-sigma-ma | not reproduced |
| fig5c-delta-change | reproduced |
| figs5-within-wake-rem | not reproduced |
| fig5c-wake-movement | blocked (data missing) |
| updown-durations | blocked (methods underspecified) |
| fig3c-slow-oscillation-spindles-over-sleep | blocked (methods underspecified) |
| fig4ii-iii-updown-spindles-within-packets | blocked (methods underspecified) |
| fig2de-up-state-sequences | blocked (methods underspecified) |
| fig3f-4e-within-up-rates | blocked (methods underspecified) |

Of 39 claims, 8 are reproduced, 9 partially reproduced, 15 not reproduced and 7 blocked.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

**Blocked claims.** UP and DOWN states, spindles and the autocorrelogram decay fit are detected or fitted by procedures described in the Supplemental Experimental Procedures, which could not be retrieved; the main text names the approach (spike and LFP criteria after Vyazovskiy et al. 2009; spindles "after filtering and integration") without thresholds, so the procedures are not fixed for this attempt. The movement of the animal during WAKE is not in the dandiset (no accelerometer, video or EMG).

**How the analysis differs from the paper's.**

- 21 of the paper's 27 sessions and 10 of its 11 rats: the dandiset holds 22 sessions, and one (Dino-072114) has unusable spike times. 812 pE and 89 pI cells, against 995 and 126.
- The states are the dandiset's table (Awake, Non-REM, REM). Microarousals are derived from Awake epochs under 40 s between two Non-REM epochs, without the paper's EMG and theta criteria. WAKE-SLEEP cycles, packets and episodes are rebuilt from the Results' definitions; the paper's Table 1 was not available.
- Time courses use 10 normalized bins, packets and episodes of at least 20 s, and group means normalized within each interval; the paper does not give its binning. Variants with 5 and 20 bins and a 60 s minimum are shown.
- LFP is one channel per session (the electrode the conversion flags as spindle reference), in raw units, with power from 2 s windows; the LFP of Dino-061914 is left out as its samples are scrambled.
- Statistics are over cells, or cell and interval pairs, pooled across sessions, as the paper's per-cell statistics; the per-subject results are printed for each claim. These pooled tests treat cells of one session, and bins of one interval, as independent. Tests with one value per cycle, triplet or session are shown for fig3-first-last-packet, fig3e, fig4iv and the fig5b triplets; they weaken fig3e (sextiles 4 and 5) and the packet half of fig4iv (sextile 6), and do not change an outcome, which follows the criteria. No random numbers are used.
- The main numbers of every tested claim are recomputed without the session whose state labels, and without the seven sessions whose LFP, could not be checked against the spikes. No judgment of a claim that uses the checked stream changes; without the seven sessions two spike-only claims, fig3-first-last-packet and fig2c-acg-peaks, would be judged differently.

**What was found about the dandiset.**

- The dandiset description and the paper give 11 animals and 27 sessions; the files hold 10 animals and 22 sessions, and 945 units (826 excitatory, 96 inhibitory, 23 unknown) where the description gives 995 pE and 126 pI.
- Dino-072114: spike times are whole seconds divided by 20000 (8451 distinct values, from 0 to 0.4252 s, in a 7959 s recording).
- Dino-061914: the LFP has 125 columns for 128 electrodes, lasts 331.7 min against 291.9 min of states and spikes, and its samples are scrambled (lag-1 autocorrelation 0.28, against 0.99 or more elsewhere).
- The state table has no microarousal label, though the paper scores MAs as a state; the derived MAs are rarer than the paper reports (391 in 21 sessions against 790 in 27), and only 49 of 534 REM epochs are followed directly by Non-REM.
- The LFP is stored as int16 with unit volts and conversion NaN, so it cannot be put in volts. Every electrode's location is "unknown" (the electrode groups carry a region), and every unit's region is "CTX" or "unknown".
- 23 units of BWRat21-121613 have cell type and region "unknown", with no description.
- The state labels are consistent with the spikes in 20 of 21 sessions (not established for BWRat21-121813); the best match is often with the table shifted 5 to 60 s earlier. The LFP is consistent with the spikes at 50 ms precision in 14 of 21 sessions.